# Sohoj — Tox21 Drug Toxicity Prediction
### Research-grade, restart-safe ML/DL notebook for Google Colab, Anaconda/JupyterLab, CPU and GPU

এই notebook-এ নিচের model গুলো রাখা হয়েছে:

1. **Random Forest** — `Ultimate_Final.ipynb`-এর preprocessing ও model parameters অপরিবর্তিত
2. **Extra Trees** — `Ultimate_Final.ipynb`-এর preprocessing ও model parameters অপরিবর্তিত
3. **XGBoost** — `Ultimate_Final.ipynb`-এর preprocessing ও model parameters অপরিবর্তিত
4. **SVM with Tanimoto kernel**
5. **DeepTox-style Multi-task DNN**
6. **Multitask CapsNet with two RBM layers and dynamic routing**
7. **DenseNet121 + SVM Tanimoto hybrid**
8. **Multi-channel 2D CNN**
9. **Compact MTL-DNN**
10. **Validation-weighted soft-voting Ensemble**

**R-GCN সম্পূর্ণভাবে বাদ দেওয়া হয়েছে।** Dataset split: **70% Train, 10% Validation, 20% Test**। Final evaluation metric: **Mean AUC-ROC** এবং **Mean Accuracy**।

> **গুরুত্বপূর্ণ বৈজ্ঞানিক নোট:** `Mean AUC > 0.90` একটি research target; notebook কোনো result hard-code বা fabricate করে না। প্রকৃত score dataset, split, hardware, package version এবং optimization-এর উপর নির্ভর করবে। Supplied papers-এর যেসব proprietary/external component (ChemAxon, JCompoundMapper, PaDEL, AutoDock-specific values) সরাসরি reproducible নয়, সেখানে স্পষ্টভাবে documented open-source RDKit approximation ব্যবহার করা হয়েছে।


## 0. Automatic environment setup

এই cell package check করবে। কোনো package missing থাকলে Bangla message দেখিয়ে install করবে। Colab-এ GPU থাকলে GPU, না থাকলে CPU ব্যবহার হবে।


In [ ]:
import os, sys, subprocess, importlib.util, platform

os.environ.setdefault("PIP_DISABLE_PIP_VERSION_CHECK", "1")

PACKAGE_MAP = {
    "numpy": "numpy",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "seaborn": "seaborn",
    "sklearn": "scikit-learn",
    "joblib": "joblib",
    "psutil": "psutil",
    "rdkit": "rdkit",
    "xgboost": "xgboost",
    "torch": "torch",
    "torchvision": "torchvision",
    "PIL": "pillow",
    "imblearn": "imbalanced-learn",
    "tqdm": "tqdm",
    "dill": "dill",
}

missing = [pip_name for module_name, pip_name in PACKAGE_MAP.items()
           if importlib.util.find_spec(module_name) is None]

if missing:
    print("⚠️ নিচের package পাওয়া যায়নি:", ", ".join(missing))
    print("⏳ প্রয়োজনীয় package install করা হচ্ছে...")
    cmd = [sys.executable, "-m", "pip", "install", "-q", *missing]
    try:
        subprocess.check_call(cmd)
        print("✅ সব missing package install সম্পন্ন হয়েছে।")
    except subprocess.CalledProcessError as exc:
        print("❌ কিছু package install হয়নি। Internet connection এবং permission check করুন।")
        raise RuntimeError("Package installation failed") from exc
else:
    print("✅ প্রয়োজনীয় সব package আগে থেকেই installed আছে।")

print("Python:", sys.version.split()[0], "| OS:", platform.platform())


## 1. Imports, reproducibility and automatic CPU/GPU selection


In [ ]:
import os, gc, re, json, math, time, random, hashlib, warnings, shutil
from pathlib import Path
from datetime import datetime
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib, psutil, dill
from tqdm.auto import tqdm
from IPython.display import display, Markdown

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.decomposition import PCA
from sklearn.metrics import roc_auc_score, accuracy_score, confusion_matrix
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.svm import SVC
from sklearn.neural_network import BernoulliRBM

from xgboost import XGBClassifier

from rdkit import Chem, DataStructs, RDLogger, RDConfig
from rdkit.Chem import (
    AllChem, MACCSkeys, Descriptors, Draw, Crippen, Lipinski,
    rdMolDescriptors, ChemicalFeatures
)
from rdkit.Chem.MolStandardize import rdMolStandardize

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

try:
    from torchvision.models import densenet121, DenseNet121_Weights
    from torchvision import transforms
    TORCHVISION_AVAILABLE = True
except Exception as exc:
    TORCHVISION_AVAILABLE = False
    print("⚠️ torchvision import করা যায়নি:", exc)

warnings.filterwarnings("ignore")
RDLogger.DisableLog("rdApp.*")

SEED = 42
THRESHOLD = 0.50
CV_FOLDS = 3
TRAIN_RATIO, VALID_RATIO, TEST_RATIO = 0.70, 0.10, 0.20
N_BITS, RADIUS = 2048, 2
TARGET_MEAN_AUC = 0.90

os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
RAM_GB = psutil.virtual_memory().total / (1024**3)
CPU_COUNT = max(1, os.cpu_count() or 1)
N_JOBS = max(1, min(CPU_COUNT - 1, 12))

if RAM_GB < 10:
    RESOURCE_MODE = "low"
elif RAM_GB < 20:
    RESOURCE_MODE = "standard"
else:
    RESOURCE_MODE = "high"

BATCH_TABULAR = 128 if RESOURCE_MODE == "low" else 256
BATCH_IMAGE = 16 if RESOURCE_MODE == "low" else (32 if RESOURCE_MODE == "standard" else 64)
NUM_WORKERS = 0 if platform.system().lower().startswith("win") else min(4, CPU_COUNT // 2)

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)

print("=" * 76)
print("✅ Reproducibility seed:", SEED)
print("✅ Computing device:", DEVICE)
if DEVICE.type == "cuda":
    print("✅ GPU:", torch.cuda.get_device_name(0))
else:
    print("ℹ️ GPU পাওয়া যায়নি; CPU-তে code চলবে।")
print(f"✅ RAM: {RAM_GB:.1f} GB | CPU threads: {CPU_COUNT} | Mode: {RESOURCE_MODE}")
print("=" * 76)


## 2. Project folders, dataset auto-detection and persistent checkpoints

- Colab + Google Drive
- Colab local runtime
- Windows/Linux Anaconda/JupyterLab
- Uploaded `/mnt/data` dataset

সব environment-এর জন্য path auto-detect করার চেষ্টা করা হবে।


In [ ]:
# Optional: নিজের dataset path force করতে environment variable set করতে পারেন:
# os.environ["TOX21_DATA_PATH"] = r"D:\Drug_Toxicity\datasets\tox21.csv"

candidate_data_paths = []
if os.environ.get("TOX21_DATA_PATH"):
    candidate_data_paths.append(Path(os.environ["TOX21_DATA_PATH"]))

candidate_data_paths += [
    Path("/content/drive/MyDrive/Drug_Toxicity/datasets/tox21.csv"),
    Path("/content/drive/MyDrive/Drug_Toxicity/datasets/tox21(4).csv"),
    Path("/content/tox21.csv"),
    Path("/content/tox21(4).csv"),
    Path("/mnt/data/tox21(4).csv"),
    Path.cwd() / "datasets" / "tox21.csv",
    Path.cwd() / "datasets" / "tox21(4).csv",
    Path.cwd() / "tox21.csv",
    Path.cwd() / "tox21(4).csv",
]

DATA_PATH = next((p for p in candidate_data_paths if p.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError(
        "tox21.csv পাওয়া যায়নি। TOX21_DATA_PATH set করুন অথবা dataset notebook-এর পাশে রাখুন।"
    )

# Persistent project root: Drive পাওয়া গেলে Drive, না হলে local folder.
if Path("/content/drive/MyDrive").exists():
    PROJECT_ROOT = Path("/content/drive/MyDrive/Drug_Toxicity")
elif (Path.cwd() / "datasets").exists():
    PROJECT_ROOT = Path.cwd()
else:
    PROJECT_ROOT = Path.cwd() / "Drug_Toxicity_Sohoj"

OUTPUT_DIR = PROJECT_ROOT / "outputs" / "Sohoj"
MODEL_DIR = PROJECT_ROOT / "models" / "Sohoj"
CACHE_DIR = PROJECT_ROOT / "cache" / "Sohoj"
CHECKPOINT_DIR = PROJECT_ROOT / "checkpoints" / "Sohoj"
FIGURE_DIR = OUTPUT_DIR / "figures"
LOG_DIR = PROJECT_ROOT / "logs" / "Sohoj"

for folder in [OUTPUT_DIR, MODEL_DIR, CACHE_DIR, CHECKPOINT_DIR, FIGURE_DIR, LOG_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

RAW_SHA256 = hashlib.sha256(DATA_PATH.read_bytes()).hexdigest()
RUN_ID = f"{RAW_SHA256[:10]}_seed{SEED}_split70_10_20"
SESSION_PATH = CACHE_DIR / f"sohoj_runtime_{RUN_ID}.pkl"
Path("sohoj_session_path.txt").write_text(str(SESSION_PATH.resolve()), encoding="utf-8")

print("✅ Dataset:", DATA_PATH)
print("✅ Project root:", PROJECT_ROOT)
print("✅ Output folder:", OUTPUT_DIR)
print("✅ Checkpoint folder:", CHECKPOINT_DIR)
print("✅ Runtime session:", SESSION_PATH)


## 3. Load dataset and strict schema validation


In [ ]:
EXPECTED_TARGETS = [
    "NR-AR", "NR-AR-LBD", "NR-AhR", "NR-Aromatase",
    "NR-ER", "NR-ER-LBD", "NR-PPAR-gamma",
    "SR-ARE", "SR-ATAD5", "SR-HSE", "SR-MMP", "SR-p53"
]

# Same aliases used only when a dataset uses alternate punctuation.
ALIASES = {
    "NR.AhR": "NR-AhR", "NR.AR": "NR-AR", "NR.AR.LBD": "NR-AR-LBD",
    "NR.Aromatase": "NR-Aromatase", "NR.ER": "NR-ER", "NR.ER.LBD": "NR-ER-LBD",
    "NR.PPAR.gamma": "NR-PPAR-gamma", "SR.ARE": "SR-ARE", "SR.ATAD5": "SR-ATAD5",
    "SR.HSE": "SR-HSE", "SR.MMP": "SR-MMP", "SR.p53": "SR-p53"
}

df_raw = pd.read_csv(DATA_PATH).rename(columns=ALIASES)
required = set(EXPECTED_TARGETS + ["smiles"])
missing_cols = sorted(required - set(df_raw.columns))
if missing_cols:
    raise ValueError(f"Dataset-এ required column missing: {missing_cols}")

if "mol_id" not in df_raw.columns:
    df_raw["mol_id"] = [f"MOL_{i:06d}" for i in range(len(df_raw))]

for col in EXPECTED_TARGETS:
    df_raw[col] = pd.to_numeric(df_raw[col], errors="coerce")
    invalid_values = set(df_raw[col].dropna().unique()) - {0.0, 1.0}
    if invalid_values:
        raise ValueError(f"{col} column-এ 0/1/NaN ছাড়া value আছে: {invalid_values}")

TARGETS = EXPECTED_TARGETS.copy()
print("✅ Dataset successfully loaded.")
print("Raw shape:", df_raw.shape)
display(df_raw.head(5).style.set_caption("Tox21 — Raw Data Preview"))


## 4. Dataset audit — missing labels and class imbalance


In [ ]:
def build_endpoint_summary(frame, targets):
    rows = []
    n = len(frame)
    for target in targets:
        s = frame[target]
        labeled = int(s.notna().sum())
        toxic = int((s == 1).sum())
        non_toxic = int((s == 0).sum())
        missing = n - labeled
        pos_rate = 100.0 * toxic / labeled if labeled else np.nan
        imbalance = non_toxic / max(toxic, 1)
        rows.append({
            "Endpoint": target, "Labeled": labeled, "Toxic (1)": toxic,
            "Non-Toxic (0)": non_toxic, "Missing": missing,
            "Missing %": 100.0 * missing / n, "Positive Rate %": pos_rate,
            "Neg:Pos": imbalance,
        })
    return pd.DataFrame(rows)

endpoint_summary_raw = build_endpoint_summary(df_raw, TARGETS)
display(
    endpoint_summary_raw.style
    .format({"Missing %": "{:.1f}", "Positive Rate %": "{:.1f}", "Neg:Pos": "{:.1f}"})
    .background_gradient(subset=["Missing %", "Neg:Pos"], cmap="YlOrRd")
    .set_caption("Endpoint-wise Missing Labels and Class Imbalance")
)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
plot_df = endpoint_summary_raw.sort_values("Missing %")
axes[0].barh(plot_df["Endpoint"], plot_df["Missing %"], color="#F4A261")
axes[0].set_title("Tox21 Dataset Challenges — Missing Labels", fontweight="bold")
axes[0].set_xlabel("Missing labels (%)")
for i, v in enumerate(plot_df["Missing %"]):
    axes[0].text(v + 0.3, i, f"{v:.1f}%", va="center", fontsize=9)

plot_df2 = endpoint_summary_raw.sort_values("Positive Rate %")
colors = ["#E76F51" if v < 5 else "#F4A261" if v < 10 else "#2A9D8F" for v in plot_df2["Positive Rate %"]]
axes[1].barh(plot_df2["Endpoint"], plot_df2["Positive Rate %"], color=colors)
axes[1].set_title("Class Imbalance — Toxic Positive Rate", fontweight="bold")
axes[1].set_xlabel("Positive toxic samples (%)")
for i, v in enumerate(plot_df2["Positive Rate %"]):
    axes[1].text(v + 0.2, i, f"{v:.1f}%", va="center", fontsize=9)

plt.tight_layout()
plt.savefig(FIGURE_DIR / "01_dataset_challenges.png", dpi=220, bbox_inches="tight")
plt.show()


In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(17, 13))
for ax, target in zip(axes.flat, TARGETS):
    counts = df_raw[target].value_counts(dropna=True).reindex([0.0, 1.0], fill_value=0)
    ax.bar(["Non-Toxic", "Toxic"], counts.values, color=["#457B9D", "#E63946"])
    ax.set_title(target, fontweight="bold")
    ax.set_ylabel("Count")
    for i, v in enumerate(counts.values):
        ax.text(i, v + max(counts.values) * 0.015, f"{int(v)}", ha="center", fontsize=8)
plt.suptitle("Per-Task Label Distribution after Missing-Label Masking", y=1.02, fontweight="bold", fontsize=16)
plt.tight_layout()
plt.savefig(FIGURE_DIR / "02_per_task_label_distribution.png", dpi=220, bbox_inches="tight")
plt.show()


## 5. Chemical preprocessing — preserved from `Ultimate_Final.ipynb`

এই অংশে **কোনো method পরিবর্তন করা হয়নি**:

- invalid SMILES removal
- largest fragment selection
- RDKit normalization
- uncharging
- canonical SMILES generation
- duplicate canonical SMILES removal

Requested 70/10/20 split পরে করা হবে; chemistry preprocessing এবং feature engineering একই রাখা হয়েছে।


In [ ]:
# === Preserved preprocessing logic from Ultimate_Final.ipynb ===
largest_fragment = rdMolStandardize.LargestFragmentChooser()
normalizer = rdMolStandardize.Normalizer()
uncharger = rdMolStandardize.Uncharger()

def standardize_smiles(smiles):
    try:
        mol = Chem.MolFromSmiles(str(smiles))
        if mol is None:
            return None, None, "invalid_parse"
        mol = largest_fragment.choose(mol)
        mol = normalizer.normalize(mol)
        mol = uncharger.uncharge(mol)
        Chem.SanitizeMol(mol)
        can = Chem.MolToSmiles(mol, canonical=True)
        mol2 = Chem.MolFromSmiles(can)
        return can, mol2, "ok"
    except Exception as e:
        return None, None, str(e)


In [ ]:
CURATED_PATH = CACHE_DIR / f"curated_{RUN_ID}.csv"
INVALID_PATH = OUTPUT_DIR / "invalid_smiles.csv"

if CURATED_PATH.exists():
    print("♻️ আগে তৈরি করা cleaned dataset পাওয়া গেছে; cache থেকে load হচ্ছে...")
    df = pd.read_csv(CURATED_PATH)
    df["mol"] = [Chem.MolFromSmiles(s) for s in df["canonical_smiles"]]
else:
    df = df_raw.copy()
    records = []
    for smi in tqdm(df["smiles"], desc="SMILES standardization"):
        can, mol, status = standardize_smiles(smi)
        records.append((can, mol, status))

    df["canonical_smiles"] = [r[0] for r in records]
    df["mol"] = [r[1] for r in records]
    df["standardize_status"] = [r[2] for r in records]

    invalid_df = df[df["mol"].isna()].copy()
    invalid_df.to_csv(INVALID_PATH, index=False)

    print("Before cleaning:", df.shape)
    print("Invalid SMILES:", int(df["mol"].isna().sum()))

    df = df[df["mol"].notna()].copy()
    dup_count = int(df.duplicated("canonical_smiles").sum())
    print("Duplicate canonical SMILES:", dup_count)

    df = df.drop_duplicates("canonical_smiles").reset_index(drop=True)
    save_cols = ["mol_id", "smiles", "canonical_smiles", *TARGETS]
    df[save_cols].to_csv(CURATED_PATH, index=False)

print("✅ Chemical cleaning সম্পন্ন।")
print("After cleaning:", df.shape)


In [ ]:
show_n = min(12, len(df))
show_idx = np.linspace(0, len(df) - 1, show_n, dtype=int)
show_mols = [df.iloc[i]["mol"] for i in show_idx]
legends = [str(df.iloc[i]["mol_id"]) for i in show_idx]
img = Draw.MolsToGridImage(show_mols, molsPerRow=4, subImgSize=(280, 220), legends=legends)
display(img)


## 6. Feature engineering — preserved from `Ultimate_Final.ipynb`

Classical models-এর জন্য exactly same representation:

- Morgan ECFP4: radius = 2, 2048 bits
- MACCS keys: 167 bits
- All RDKit 2D descriptors
- descriptor `inf` → `NaN`
- train-only median imputation and standard scaling


In [ ]:
# === Preserved feature functions from Ultimate_Final.ipynb ===
def morgan_fp(mol, n_bits=N_BITS, radius=RADIUS):
    arr = np.zeros((n_bits,), dtype=np.int8)
    fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits=n_bits)
    DataStructs.ConvertToNumpyArray(fp, arr)
    return arr

def maccs_fp(mol):
    arr = np.zeros((167,), dtype=np.int8)
    fp = MACCSkeys.GenMACCSKeys(mol)
    DataStructs.ConvertToNumpyArray(fp, arr)
    return arr

descriptor_names = [name for name, _ in Descriptors._descList]
descriptor_funcs = [func for _, func in Descriptors._descList]

def rdkit_descriptors(mol):
    vals = []
    for func in descriptor_funcs:
        try:
            vals.append(float(func(mol)))
        except Exception:
            vals.append(np.nan)
    return vals


In [ ]:
FEATURE_DIR = CACHE_DIR / f"ultimate_features_{RUN_ID}"
FEATURE_DIR.mkdir(parents=True, exist_ok=True)
MORGAN_PATH = FEATURE_DIR / "morgan_2048.npy"
MACCS_PATH = FEATURE_DIR / "maccs_167.npy"
DESC_PATH = FEATURE_DIR / "rdkit_descriptors.npy"
FEATURE_PROGRESS = FEATURE_DIR / "progress.json"

n_samples = len(df)
n_desc = len(descriptor_names)
chunk_size = 128

feature_cache_complete = (
    all(p.exists() for p in [MORGAN_PATH, MACCS_PATH, DESC_PATH])
    and FEATURE_PROGRESS.exists()
    and json.loads(FEATURE_PROGRESS.read_text(encoding="utf-8")).get("done", False)
)

if feature_cache_complete:
    print("♻️ আগে তৈরি করা chemical features cache থেকে load হচ্ছে...")
else:
    print("⏳ Morgan, MACCS এবং RDKit descriptor তৈরি হচ্ছে...")
    xm = np.lib.format.open_memmap(MORGAN_PATH, mode="w+", dtype="float32", shape=(n_samples, N_BITS))
    xk = np.lib.format.open_memmap(MACCS_PATH, mode="w+", dtype="float32", shape=(n_samples, 167))
    xd = np.lib.format.open_memmap(DESC_PATH, mode="w+", dtype="float32", shape=(n_samples, n_desc))

    start = 0
    if FEATURE_PROGRESS.exists():
        try:
            start = int(json.loads(FEATURE_PROGRESS.read_text())["completed"])
        except Exception:
            start = 0

    # If a prior interrupted memmap was recreated, safely restart from zero.
    if start > 0:
        print("ℹ️ Feature file নতুনভাবে খোলা হয়েছে; consistency-এর জন্য zero থেকে rebuild হবে।")
        start = 0

    for begin in tqdm(range(start, n_samples, chunk_size), desc="Chemical feature generation"):
        end = min(begin + chunk_size, n_samples)
        for i in range(begin, end):
            mol = df.iloc[i]["mol"]
            xm[i] = morgan_fp(mol)
            xk[i] = maccs_fp(mol)
            xd[i] = np.asarray(rdkit_descriptors(mol), dtype=np.float32)
        xm.flush(); xk.flush(); xd.flush()
        FEATURE_PROGRESS.write_text(json.dumps({"completed": end}), encoding="utf-8")

    FEATURE_PROGRESS.write_text(json.dumps({"completed": n_samples, "done": True}), encoding="utf-8")
    print("✅ Chemical feature generation সম্পন্ন।")

X_morgan = np.load(MORGAN_PATH, mmap_mode="r")
X_maccs = np.load(MACCS_PATH, mmap_mode="r")
X_desc_arr = np.load(DESC_PATH, mmap_mode="r")
X_desc_raw = pd.DataFrame(np.asarray(X_desc_arr), columns=descriptor_names).replace([np.inf, -np.inf], np.nan)
X_fp = np.hstack([np.asarray(X_morgan), np.asarray(X_maccs)]).astype(np.float32)
Y = df[TARGETS].values.astype(float)

print("Morgan:", X_morgan.shape)
print("MACCS:", X_maccs.shape)
print("Raw descriptors:", X_desc_raw.shape)
print("Targets:", Y.shape)


## 7. Dataset split — 70% Train, 10% Validation, 20% Test

Row-level stratification একই রাখা হয়েছে: কোনো molecule-এ অন্তত একটি positive toxicity label আছে কি না। Test set model selection বা ensemble weighting-এ ব্যবহার হবে না।


In [ ]:
SPLIT_PATH = CACHE_DIR / f"split_indices_{RUN_ID}.npz"
row_stratify = (np.nan_to_num(Y, nan=0.0).sum(axis=1) > 0).astype(int)
all_indices = np.arange(len(df))

if SPLIT_PATH.exists():
    split_data = np.load(SPLIT_PATH)
    train_idx = split_data["train_idx"]
    val_idx = split_data["val_idx"]
    test_idx = split_data["test_idx"]
    print("♻️ Frozen split cache থেকে load হয়েছে।")
else:
    stratify_all = row_stratify if len(np.unique(row_stratify)) == 2 else None
    train_val_idx, test_idx = train_test_split(
        all_indices,
        test_size=TEST_RATIO,
        random_state=SEED,
        stratify=stratify_all,
    )

    # 10% of whole data = 12.5% of the remaining 80%.
    val_fraction_of_train_val = VALID_RATIO / (TRAIN_RATIO + VALID_RATIO)
    stratify_tv = row_stratify[train_val_idx] if len(np.unique(row_stratify[train_val_idx])) == 2 else None
    train_idx, val_idx = train_test_split(
        train_val_idx,
        test_size=val_fraction_of_train_val,
        random_state=SEED,
        stratify=stratify_tv,
    )
    np.savez_compressed(SPLIT_PATH, train_idx=train_idx, val_idx=val_idx, test_idx=test_idx)

Y_train, Y_val, Y_test = Y[train_idx], Y[val_idx], Y[test_idx]

print("✅ ডেটাসেট ৩ ভাগ করা হয়েছে।")
print(f"Train      : {len(train_idx):5d} ({100*len(train_idx)/len(df):.2f}%)")
print(f"Validation : {len(val_idx):5d} ({100*len(val_idx)/len(df):.2f}%)")
print(f"Test       : {len(test_idx):5d} ({100*len(test_idx)/len(df):.2f}%)")


In [ ]:
def split_endpoint_counts(indices, split_name):
    rows = []
    part = df.iloc[indices]
    for target in TARGETS:
        rows.append({
            "Split": split_name,
            "Endpoint": target,
            "Non-Toxic": int((part[target] == 0).sum()),
            "Toxic": int((part[target] == 1).sum()),
            "Missing": int(part[target].isna().sum()),
        })
    return pd.DataFrame(rows)

split_dist = pd.concat([
    split_endpoint_counts(train_idx, "Train"),
    split_endpoint_counts(val_idx, "Validation"),
    split_endpoint_counts(test_idx, "Test"),
], ignore_index=True)

display(split_dist.head(12).style.set_caption("Frozen Split — Endpoint Distribution Preview"))


## 8. Train-only imputation and scaling — preserved from `Ultimate_Final.ipynb`

Imputer এবং scaler শুধুমাত্র 70% training split-এ fit হবে। Validation এবং Test-এ transform only।


In [ ]:
# === Preserved preprocessing function, extended only to include validation transform ===
def build_chemical_features(fit_indices, transform_indices):
    """Fit imputer/scaler on fit_indices and transform transform_indices."""
    imp = SimpleImputer(strategy="median")
    scaler = StandardScaler()

    desc_fit = imp.fit_transform(X_desc_raw.iloc[fit_indices])
    desc_fit = scaler.fit_transform(desc_fit)

    desc_transform = imp.transform(X_desc_raw.iloc[transform_indices])
    desc_transform = scaler.transform(desc_transform)

    X_fit = np.hstack([X_fp[fit_indices], desc_fit]).astype(np.float32)
    X_transform = np.hstack([X_fp[transform_indices], desc_transform]).astype(np.float32)
    return X_fit, X_transform, imp, scaler

X_train_chem, X_val_chem, desc_imputer, desc_scaler = build_chemical_features(train_idx, val_idx)
desc_test = desc_scaler.transform(desc_imputer.transform(X_desc_raw.iloc[test_idx]))
X_test_chem = np.hstack([X_fp[test_idx], desc_test]).astype(np.float32)

joblib.dump({"imputer": desc_imputer, "scaler": desc_scaler}, MODEL_DIR / "ultimate_preprocessing.joblib")
np.save(FEATURE_DIR / "X_train_chem.npy", X_train_chem)
np.save(FEATURE_DIR / "X_val_chem.npy", X_val_chem)
np.save(FEATURE_DIR / "X_test_chem.npy", X_test_chem)

print("✅ Train-only imputation and scaling সম্পন্ন।")
print("Train feature shape     :", X_train_chem.shape)
print("Validation feature shape:", X_val_chem.shape)
print("Test feature shape      :", X_test_chem.shape)


## 9. Evaluation, checkpoint and overfitting/underfitting helpers

- Missing labels evaluation থেকে বাদ যাবে
- AUC এবং Accuracy endpoint-wise ও mean আকারে report হবে
- Train–Validation AUC gap দিয়ে fitting status report হবে
- প্রতিটি endpoint/model শেষ হলে prediction ও model disk-এ save হবে


In [ ]:
def slugify(text):
    return re.sub(r"[^a-z0-9]+", "_", text.lower()).strip("_")

def valid_mask(y_true, y_prob=None):
    y_true = np.asarray(y_true, dtype=float)
    mask = np.isfinite(y_true)
    if y_prob is not None:
        mask &= np.isfinite(np.asarray(y_prob, dtype=float))
    return mask

def safe_auc(y_true, y_prob):
    mask = valid_mask(y_true, y_prob)
    yt = np.asarray(y_true, dtype=float)[mask]
    yp = np.asarray(y_prob, dtype=float)[mask]
    if len(yt) == 0 or len(np.unique(yt)) < 2:
        return np.nan
    return float(roc_auc_score(yt, yp))

def safe_acc(y_true, y_prob, threshold=THRESHOLD):
    mask = valid_mask(y_true, y_prob)
    yt = np.asarray(y_true, dtype=float)[mask]
    yp = np.asarray(y_prob, dtype=float)[mask]
    if len(yt) == 0:
        return np.nan
    return float(accuracy_score(yt.astype(int), (yp >= threshold).astype(int)))

def mean_metric(values):
    arr = pd.to_numeric(pd.Series(values), errors="coerce").to_numpy(dtype=float)
    return float(np.nanmean(arr)) if np.isfinite(arr).any() else np.nan

def prior_probability(y):
    y = np.asarray(y, dtype=float)
    return float(np.nanmean(y)) if np.isfinite(y).any() else 0.0

def predict_positive_probability(model, X):
    if hasattr(model, "predict_proba"):
        proba = np.asarray(model.predict_proba(X))
        if proba.ndim == 1:
            return proba.astype(float)
        if proba.shape[1] == 1:
            return np.zeros(len(X), dtype=float)
        return proba[:, 1].astype(float)
    if hasattr(model, "decision_function"):
        score = np.asarray(model.decision_function(X), dtype=float)
        score = np.clip(score, -50, 50)
        return 1.0 / (1.0 + np.exp(-score))
    return np.asarray(model.predict(X), dtype=float)

def endpoint_metric_table(P_train, P_val, P_test):
    rows = []
    for j, target in enumerate(TARGETS):
        rows.append({
            "Endpoint": target,
            "Train AUC": safe_auc(Y_train[:, j], P_train[:, j]),
            "Validation AUC": safe_auc(Y_val[:, j], P_val[:, j]),
            "Test AUC": safe_auc(Y_test[:, j], P_test[:, j]),
            "Train Accuracy": safe_acc(Y_train[:, j], P_train[:, j]),
            "Validation Accuracy": safe_acc(Y_val[:, j], P_val[:, j]),
            "Test Accuracy": safe_acc(Y_test[:, j], P_test[:, j]),
            "Test Labeled": int(np.isfinite(Y_test[:, j]).sum()),
            "Test Toxic": int(np.nansum(Y_test[:, j] == 1)),
            "Test Non-Toxic": int(np.nansum(Y_test[:, j] == 0)),
        })
    return pd.DataFrame(rows)

def fitting_status(train_auc, val_auc):
    if not np.isfinite(train_auc) or not np.isfinite(val_auc):
        return "Insufficient data"
    gap = train_auc - val_auc
    if train_auc >= 0.85 and gap > 0.08:
        return "Overfitting"
    if train_auc < 0.72 and val_auc < 0.72:
        return "Underfitting"
    if val_auc - train_auc > 0.05:
        return "Split-unstable"
    return "Well-fitted"

def register_model_result(model_name, P_train, P_val, P_test, history=None, notes=""):
    P_train = np.clip(np.nan_to_num(P_train, nan=0.0), 0.0, 1.0)
    P_val = np.clip(np.nan_to_num(P_val, nan=0.0), 0.0, 1.0)
    P_test = np.clip(np.nan_to_num(P_test, nan=0.0), 0.0, 1.0)

    model_dir = OUTPUT_DIR / slugify(model_name)
    model_dir.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(model_dir / "predictions.npz", train=P_train, validation=P_val, test=P_test)

    per_endpoint = endpoint_metric_table(P_train, P_val, P_test)
    per_endpoint.to_csv(model_dir / "per_endpoint_metrics.csv", index=False)

    train_auc = mean_metric(per_endpoint["Train AUC"])
    val_auc = mean_metric(per_endpoint["Validation AUC"])
    test_auc = mean_metric(per_endpoint["Test AUC"])
    train_acc = mean_metric(per_endpoint["Train Accuracy"])
    val_acc = mean_metric(per_endpoint["Validation Accuracy"])
    test_acc = mean_metric(per_endpoint["Test Accuracy"])
    status = fitting_status(train_auc, val_auc)

    summary = {
        "Model": model_name,
        "Train Mean AUC": train_auc,
        "Validation Mean AUC": val_auc,
        "Test Mean AUC": test_auc,
        "Train Mean Accuracy": train_acc,
        "Validation Mean Accuracy": val_acc,
        "Test Mean Accuracy": test_acc,
        "Train-Validation AUC Gap": train_auc - val_auc,
        "Fitting Status": status,
        "Target AUC Reached": bool(np.isfinite(test_auc) and test_auc >= TARGET_MEAN_AUC),
        "Notes": notes,
    }
    (model_dir / "overall_summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
    if history is not None:
        (model_dir / "history.json").write_text(json.dumps(history, indent=2), encoding="utf-8")

    print("\n" + "=" * 86)
    print(f"✅ {model_name}")
    print(f"Train Mean AUC      : {train_auc:.4f}")
    print(f"Validation Mean AUC : {val_auc:.4f}")
    print(f"Test Mean AUC       : {test_auc:.4f}")
    print(f"Test Mean Accuracy  : {test_acc:.4f}")
    print(f"Fitting Status      : {status}")
    print("=" * 86)
    display(
        per_endpoint[["Endpoint", "Train AUC", "Validation AUC", "Test AUC", "Test Accuracy", "Test Toxic", "Test Non-Toxic"]]
        .style.format({
            "Train AUC": "{:.4f}", "Validation AUC": "{:.4f}",
            "Test AUC": "{:.4f}", "Test Accuracy": "{:.4f}"
        })
        .background_gradient(subset=["Test AUC"], cmap="RdYlGn", vmin=0.5, vmax=1.0)
        .set_caption(f"{model_name} — Endpoint-wise Test Performance")
    )
    return summary, per_endpoint


def rehydrate_runtime():
    """Reload large arrays and RDKit molecules that are intentionally excluded from dill session files."""
    global df, X_morgan, X_maccs, X_desc_arr, X_desc_raw, X_fp, Y
    global train_idx, val_idx, test_idx, Y_train, Y_val, Y_test
    global X_train_chem, X_val_chem, X_test_chem, SOHOJ_READY

    df = pd.read_csv(CURATED_PATH)
    df["mol"] = [Chem.MolFromSmiles(s) for s in df["canonical_smiles"]]
    X_morgan = np.load(MORGAN_PATH, mmap_mode="r")
    X_maccs = np.load(MACCS_PATH, mmap_mode="r")
    X_desc_arr = np.load(DESC_PATH, mmap_mode="r")
    X_desc_raw = pd.DataFrame(np.asarray(X_desc_arr), columns=descriptor_names).replace([np.inf, -np.inf], np.nan)
    X_fp = np.hstack([np.asarray(X_morgan), np.asarray(X_maccs)]).astype(np.float32)
    Y = df[TARGETS].values.astype(float)

    split_data = np.load(SPLIT_PATH)
    train_idx, val_idx, test_idx = split_data["train_idx"], split_data["val_idx"], split_data["test_idx"]
    Y_train, Y_val, Y_test = Y[train_idx], Y[val_idx], Y[test_idx]
    X_train_chem = np.load(FEATURE_DIR / "X_train_chem.npy")
    X_val_chem = np.load(FEATURE_DIR / "X_val_chem.npy")
    X_test_chem = np.load(FEATURE_DIR / "X_test_chem.npy")

    # Optional heavy caches are reloaded only when their path variables exist in the restored session.
    if "GRID_PATH" in globals() and Path(GRID_PATH).exists():
        globals()["X_grids"] = np.load(GRID_PATH, mmap_mode="r")
    if "DENSE_PATH" in globals() and Path(DENSE_PATH).exists():
        globals()["X_dense_all"] = np.load(DENSE_PATH, mmap_mode="r")
    for var_name, path_name in [
        ("K_tr", "KERNEL_DIR/ecfp_train.npy"),
        ("K_va", "KERNEL_DIR/ecfp_validation.npy"),
        ("K_te", "KERNEL_DIR/ecfp_test.npy"),
    ]:
        if "KERNEL_DIR" in globals():
            filename = path_name.split("/")[-1]
            fp = Path(KERNEL_DIR) / filename
            if fp.exists(): globals()[var_name] = np.load(fp, mmap_mode="r")
    if "DENSE_KERNEL_DIR" in globals():
        optional_kernel_files = {
            "Kv_tr": "visual_train.npy", "Kv_va": "visual_validation.npy", "Kv_te": "visual_test.npy",
            "Khy_tr": "hybrid_train.npy", "Khy_va": "hybrid_validation.npy", "Khy_te": "hybrid_test.npy",
        }
        for var_name, filename in optional_kernel_files.items():
            fp = Path(DENSE_KERNEL_DIR) / filename
            if fp.exists(): globals()[var_name] = np.load(fp, mmap_mode="r")

    SOHOJ_READY = True
    print("✅ Large arrays এবং RDKit molecules persistent cache থেকে rehydrate হয়েছে।")


def save_runtime_session():
    # Large arrays/memmaps and C++ wrapper objects are intentionally excluded.
    # They are reloaded by rehydrate_runtime() from persistent cache after a kernel restart.
    explicit_skip = {
        "largest_fragment", "normalizer", "uncharger", "descriptor_funcs",
        "FEATURE_FACTORY", "PTABLE", "dense_backbone", "dense_transform", "weights",
        "df", "X_morgan", "X_maccs", "X_desc_arr", "X_desc_raw", "X_fp", "Y",
        "Y_train", "Y_val", "Y_test", "X_train_chem", "X_val_chem", "X_test_chem",
        "X_dense_all", "X_grids", "K_tr", "K_va", "K_te", "Kv_tr", "Kv_va", "Kv_te",
        "Khy_tr", "Khy_va", "Khy_te", "xm", "xk", "xd", "grid_mem", "dense_mem", "out", "split_data", "cached",
        "deeptox_model", "deeptox_optimizer", "deeptox_scheduler",
        "mtl_model", "mtl_optimizer", "mtl_scheduler",
        "fig", "axes", "ax", "dense_backbone", "dense_transform"
    }
    held = {}
    for key, value in list(globals().items()):
        is_large_array = isinstance(value, np.ndarray) and value.nbytes > 8 * 1024 * 1024
        is_torch_state = (
            "torch" in globals() and (
                isinstance(value, torch.Tensor) or
                isinstance(value, nn.Module) or
                isinstance(value, torch.optim.Optimizer)
            )
        )
        if key in explicit_skip or isinstance(value, (np.memmap, np.lib.npyio.NpzFile)) or is_large_array or is_torch_state:
            held[key] = globals().pop(key)

    temp_path = SESSION_PATH.with_suffix(SESSION_PATH.suffix + ".tmp")
    try:
        if temp_path.exists(): temp_path.unlink()
        import __main__
        # Save the active notebook namespace as __main__. On recovery it is loaded
        # directly into the new kernel's __main__, so helper functions keep correct globals.
        dill.dump_module(str(temp_path), module=__main__, refimported=True)
        os.replace(temp_path, SESSION_PATH)
        Path("sohoj_session_path.txt").write_text(str(SESSION_PATH.resolve()), encoding="utf-8")
        print("💾 Runtime session checkpoint save হয়েছে।")
        return True
    except Exception as exc:
        if temp_path.exists(): temp_path.unlink()
        print("⚠️ Runtime session save করা যায়নি; model files/checkpoints তবু save আছে:", exc)
        return False
    finally:
        globals().update(held)


In [ ]:
def build_chemical_features_for_fold(fit_indices, transform_indices):
    """Exact Ultimate_Final train-only descriptor preprocessing inside a CV fold."""
    imp = SimpleImputer(strategy="median")
    scaler = StandardScaler()
    desc_fit = scaler.fit_transform(imp.fit_transform(X_desc_raw.iloc[fit_indices]))
    desc_transform = scaler.transform(imp.transform(X_desc_raw.iloc[transform_indices]))
    X_fit = np.hstack([X_fp[fit_indices], desc_fit]).astype(np.float32)
    X_transform = np.hstack([X_fp[transform_indices], desc_transform]).astype(np.float32)
    return X_fit, X_transform

def run_classical_cv(model_name, model_factory):
    cv_path = OUTPUT_DIR / slugify(model_name) / "three_fold_cv.csv"
    cv_path.parent.mkdir(parents=True, exist_ok=True)
    if cv_path.exists():
        print("♻️", model_name, "3-fold CV cache থেকে load হয়েছে।")
        return pd.read_csv(cv_path)

    row_y = (np.nan_to_num(Y_train, nan=0.0).sum(axis=1) > 0).astype(int)
    skf = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=SEED)
    fold_rows = []

    for fold, (tr_rel, va_rel) in enumerate(skf.split(np.arange(len(train_idx)), row_y), start=1):
        tr_abs, va_abs = train_idx[tr_rel], train_idx[va_rel]
        Xtr, Xva = build_chemical_features_for_fold(tr_abs, va_abs)
        Ytr, Yva = Y[tr_abs], Y[va_abs]
        Pva = np.full((len(va_rel), len(TARGETS)), np.nan, dtype=float)

        for j, target in enumerate(TARGETS):
            ytr = Ytr[:, j]
            mask = np.isfinite(ytr)
            fallback = prior_probability(ytr)
            if mask.sum() < 5 or len(np.unique(ytr[mask])) < 2:
                Pva[:, j] = fallback
                continue
            model = model_factory(ytr[mask])
            model.fit(Xtr[mask], ytr[mask].astype(int))
            Pva[:, j] = np.clip(predict_positive_probability(model, Xva), 0, 1)

        aucs = [safe_auc(Yva[:, j], Pva[:, j]) for j in range(len(TARGETS))]
        accs = [safe_acc(Yva[:, j], Pva[:, j]) for j in range(len(TARGETS))]
        fold_rows.append({
            "Model": model_name, "Fold": fold,
            "CV Mean AUC": mean_metric(aucs),
            "CV Mean Accuracy": mean_metric(accs),
        })
        print(f"{model_name} | Fold {fold}: AUC={fold_rows[-1]['CV Mean AUC']:.4f}, Accuracy={fold_rows[-1]['CV Mean Accuracy']:.4f}")

    cv_df = pd.DataFrame(fold_rows)
    cv_df.to_csv(cv_path, index=False)
    return cv_df

def run_classical_endpoint_model(model_name, model_factory, run_cv=True):
    model_output = OUTPUT_DIR / slugify(model_name)
    endpoint_dir = model_output / "endpoint_predictions"
    saved_model_dir = MODEL_DIR / slugify(model_name)
    endpoint_dir.mkdir(parents=True, exist_ok=True)
    saved_model_dir.mkdir(parents=True, exist_ok=True)

    if run_cv:
        run_classical_cv(model_name, model_factory)

    P_train = np.full((len(train_idx), len(TARGETS)), np.nan, dtype=float)
    P_val = np.full((len(val_idx), len(TARGETS)), np.nan, dtype=float)
    P_test = np.full((len(test_idx), len(TARGETS)), np.nan, dtype=float)

    for j, target in enumerate(TARGETS):
        pred_path = endpoint_dir / f"{j:02d}_{slugify(target)}.npz"
        model_path = saved_model_dir / f"{j:02d}_{slugify(target)}.joblib"

        if pred_path.exists():
            cached = np.load(pred_path)
            P_train[:, j] = cached["train"]
            P_val[:, j] = cached["validation"]
            P_test[:, j] = cached["test"]
            print(f"♻️ {model_name} | {target}: completed endpoint skip করা হলো।")
            continue

        ytr = Y_train[:, j]
        mask = np.isfinite(ytr)
        fallback = prior_probability(ytr)

        if mask.sum() < 5 or len(np.unique(ytr[mask])) < 2:
            p_tr = np.full(len(train_idx), fallback)
            p_va = np.full(len(val_idx), fallback)
            p_te = np.full(len(test_idx), fallback)
            model = None
        else:
            print(f"⏳ {model_name} | {target} train হচ্ছে...")
            model = model_factory(ytr[mask])
            model.fit(X_train_chem[mask], ytr[mask].astype(int))
            p_tr = predict_positive_probability(model, X_train_chem)
            p_va = predict_positive_probability(model, X_val_chem)
            p_te = predict_positive_probability(model, X_test_chem)
            joblib.dump(model, model_path)

        p_tr = np.clip(np.nan_to_num(p_tr, nan=fallback), 0, 1)
        p_va = np.clip(np.nan_to_num(p_va, nan=fallback), 0, 1)
        p_te = np.clip(np.nan_to_num(p_te, nan=fallback), 0, 1)
        np.savez_compressed(pred_path, train=p_tr, validation=p_va, test=p_te)
        P_train[:, j], P_val[:, j], P_test[:, j] = p_tr, p_va, p_te
        print(f"✅ {model_name} | {target} complete এবং checkpoint save হয়েছে।")

    return register_model_result(model_name, P_train, P_val, P_test)


In [ ]:
SOHOJ_READY = True
save_runtime_session()
print("✅ Base preprocessing session ready. এখন kernel restart হলেও প্রতিটি model cell নিজে session recover করতে পারবে।")


# Classical Machine Learning Models

প্রতিটি model cell restart-safe। Kernel restart হলে একই model cell আবার run করুন; completed endpoint checkpoint থেকে skip হবে।


## Model 1 — Random Forest

`Ultimate_Final.ipynb` থেকে preprocessing, 3-fold CV এবং model hyperparameters অপরিবর্তিত রাখা হয়েছে। Random Forest নিজেই bagging-based ensemble।


In [ ]:
if "SOHOJ_READY" not in globals():
    import dill
    from pathlib import Path
    pointer = Path("sohoj_session_path.txt")
    session_file = Path(pointer.read_text(encoding="utf-8").strip())
    print("♻️ Kernel restart detected; saved session load হচ্ছে...")
    import __main__
    dill.load_module(str(session_file), module=__main__)
    if "rehydrate_runtime" in globals():
        rehydrate_runtime()

def rf_factory(y):
    return RandomForestClassifier(
        n_estimators=450,
        max_depth=None,
        min_samples_leaf=2,
        max_features="sqrt",
        class_weight="balanced_subsample",
        random_state=SEED,
        n_jobs=N_JOBS
    )

rf_summary, rf_per_endpoint = run_classical_endpoint_model("Random Forest", rf_factory, run_cv=True)
save_runtime_session()


## Model 2 — Extra Trees

`Ultimate_Final.ipynb` থেকে preprocessing, 3-fold CV এবং model hyperparameters অপরিবর্তিত। এটি stronger randomized bagging-style tree ensemble।


In [ ]:
if "SOHOJ_READY" not in globals():
    import dill
    from pathlib import Path
    session_file = Path(Path("sohoj_session_path.txt").read_text(encoding="utf-8").strip())
    print("♻️ Kernel restart detected; saved session load হচ্ছে...")
    import __main__
    dill.load_module(str(session_file), module=__main__)
    if "rehydrate_runtime" in globals():
        rehydrate_runtime()

def et_factory(y):
    return ExtraTreesClassifier(
        n_estimators=500,
        max_depth=None,
        min_samples_leaf=2,
        max_features="sqrt",
        class_weight="balanced",
        random_state=SEED,
        n_jobs=N_JOBS
    )

et_summary, et_per_endpoint = run_classical_endpoint_model("Extra Trees", et_factory, run_cv=True)
save_runtime_session()


## Model 3 — XGBoost

`Ultimate_Final.ipynb` থেকে preprocessing, positive/negative ratio, 3-fold CV এবং model hyperparameters অপরিবর্তিত। এটি gradient boosting model।


In [ ]:
if "SOHOJ_READY" not in globals():
    import dill
    from pathlib import Path
    session_file = Path(Path("sohoj_session_path.txt").read_text(encoding="utf-8").strip())
    print("♻️ Kernel restart detected; saved session load হচ্ছে...")
    import __main__
    dill.load_module(str(session_file), module=__main__)
    if "rehydrate_runtime" in globals():
        rehydrate_runtime()

def xgb_factory(y):
    pos = max(np.sum(y == 1), 1)
    neg = max(np.sum(y == 0), 1)
    return XGBClassifier(
        n_estimators=350,
        max_depth=5,
        learning_rate=0.03,
        subsample=0.85,
        colsample_bytree=0.85,
        objective="binary:logistic",
        eval_metric="logloss",
        scale_pos_weight=neg / pos,
        random_state=SEED,
        n_jobs=N_JOBS,
        tree_method="hist"
    )

xgb_summary, xgb_per_endpoint = run_classical_endpoint_model("XGBoost", xgb_factory, run_cv=True)
save_runtime_session()


## Model 4 — SVM with ECFP4 Tanimoto Kernel

DeepTox paper-এর similarity-based SVM idea অনুসরণ করা হয়েছে:

- ECFP4 binary fingerprint
- exact binary Tanimoto/Jaccard kernel
- endpoint-wise SVM
- missing labels masked
- `C` train split-এর ভিতরে 3-fold CV দিয়ে select
- validation/test kernel শুধু training compounds-এর বিরুদ্ধে compute হয়


In [ ]:
if "SOHOJ_READY" not in globals():
    import dill
    from pathlib import Path
    session_file = Path(Path("sohoj_session_path.txt").read_text(encoding="utf-8").strip())
    print("♻️ Kernel restart detected; saved session load হচ্ছে...")
    import __main__
    dill.load_module(str(session_file), module=__main__)
    if "rehydrate_runtime" in globals():
        rehydrate_runtime()

def compute_tanimoto_kernel_cached(A, B, path, block_size=256):
    path = Path(path)
    if path.exists():
        print("♻️ Tanimoto kernel cache থেকে load:", path.name)
        return np.load(path, mmap_mode="r")

    A = np.asarray(A, dtype=np.float32)
    B = np.asarray(B, dtype=np.float32)
    out = np.lib.format.open_memmap(path, mode="w+", dtype="float32", shape=(len(A), len(B)))
    b_sum = B.sum(axis=1)[None, :]
    for start in tqdm(range(0, len(A), block_size), desc=f"Tanimoto {path.stem}"):
        end = min(start + block_size, len(A))
        block = A[start:end]
        inter = block @ B.T
        denom = block.sum(axis=1)[:, None] + b_sum - inter
        out[start:end] = np.divide(inter, denom, out=np.zeros_like(inter), where=denom > 0)
        out.flush()
    return np.load(path, mmap_mode="r")

KERNEL_DIR = CACHE_DIR / f"tanimoto_kernels_{RUN_ID}"
KERNEL_DIR.mkdir(parents=True, exist_ok=True)
Xtr_ecfp = np.asarray(X_morgan[train_idx], dtype=np.float32)
Xva_ecfp = np.asarray(X_morgan[val_idx], dtype=np.float32)
Xte_ecfp = np.asarray(X_morgan[test_idx], dtype=np.float32)

K_tr = compute_tanimoto_kernel_cached(Xtr_ecfp, Xtr_ecfp, KERNEL_DIR / "ecfp_train.npy")
K_va = compute_tanimoto_kernel_cached(Xva_ecfp, Xtr_ecfp, KERNEL_DIR / "ecfp_validation.npy")
K_te = compute_tanimoto_kernel_cached(Xte_ecfp, Xtr_ecfp, KERNEL_DIR / "ecfp_test.npy")
print("✅ ECFP4 Tanimoto kernels ready:", K_tr.shape, K_va.shape, K_te.shape)

save_runtime_session()


In [ ]:
def select_tanimoto_c(K, y, c_grid=None):
    c_grid = c_grid or ([1.0, 10.0] if RESOURCE_MODE == "low" else [0.5, 2.0, 8.0, 32.0])
    y = np.asarray(y).astype(int)
    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)
    best_c, best_auc = c_grid[0], -np.inf
    for c in c_grid:
        fold_scores = []
        for tr, va in skf.split(np.arange(len(y)), y):
            model = SVC(kernel="precomputed", C=c, class_weight="balanced", probability=False, cache_size=1500)
            model.fit(np.asarray(K)[np.ix_(tr, tr)], y[tr])
            score = model.decision_function(np.asarray(K)[np.ix_(va, tr)])
            fold_scores.append(safe_auc(y[va], score))
        mean_auc = mean_metric(fold_scores)
        if mean_auc > best_auc:
            best_c, best_auc = c, mean_auc
    return float(best_c), float(best_auc)

def run_tanimoto_svm(model_name, K_train, K_validation, K_test, reuse_c=None):
    model_output = OUTPUT_DIR / slugify(model_name)
    endpoint_dir = model_output / "endpoint_predictions"
    model_save_dir = MODEL_DIR / slugify(model_name)
    endpoint_dir.mkdir(parents=True, exist_ok=True)
    model_save_dir.mkdir(parents=True, exist_ok=True)

    P_train = np.zeros((len(train_idx), len(TARGETS)), dtype=float)
    P_val = np.zeros((len(val_idx), len(TARGETS)), dtype=float)
    P_test = np.zeros((len(test_idx), len(TARGETS)), dtype=float)
    selected_rows = []

    for j, target in enumerate(TARGETS):
        pred_path = endpoint_dir / f"{j:02d}_{slugify(target)}.npz"
        if pred_path.exists():
            cached = np.load(pred_path)
            P_train[:, j], P_val[:, j], P_test[:, j] = cached["train"], cached["validation"], cached["test"]
            selected_rows.append({"Endpoint": target, "C": float(cached["C"]), "CV AUC": float(cached["cv_auc"])})
            print(f"♻️ {model_name} | {target}: checkpoint load হয়েছে।")
            continue

        ytr = Y_train[:, j]
        labeled_idx = np.where(np.isfinite(ytr))[0]
        y_labeled = ytr[labeled_idx].astype(int)
        fallback = prior_probability(ytr)

        if len(np.unique(y_labeled)) < 2:
            p_tr = np.full(len(train_idx), fallback)
            p_va = np.full(len(val_idx), fallback)
            p_te = np.full(len(test_idx), fallback)
            best_c, cv_auc = 1.0, np.nan
        else:
            if reuse_c is not None and target in reuse_c:
                best_c, cv_auc = float(reuse_c[target]), np.nan
            else:
                K_labeled = np.asarray(K_train)[np.ix_(labeled_idx, labeled_idx)]
                best_c, cv_auc = select_tanimoto_c(K_labeled, y_labeled)

            print(f"⏳ {model_name} | {target} | selected C={best_c:g}")
            model = SVC(
                kernel="precomputed", C=best_c, class_weight="balanced",
                probability=False, cache_size=1500, random_state=SEED
            )
            model.fit(np.asarray(K_train)[np.ix_(labeled_idx, labeled_idx)], y_labeled)
            s_tr = model.decision_function(np.asarray(K_train)[:, labeled_idx])
            s_va = model.decision_function(np.asarray(K_validation)[:, labeled_idx])
            s_te = model.decision_function(np.asarray(K_test)[:, labeled_idx])
            p_tr = 1.0 / (1.0 + np.exp(-np.clip(s_tr, -50, 50)))
            p_va = 1.0 / (1.0 + np.exp(-np.clip(s_va, -50, 50)))
            p_te = 1.0 / (1.0 + np.exp(-np.clip(s_te, -50, 50)))
            joblib.dump(model, model_save_dir / f"{j:02d}_{slugify(target)}.joblib")

        np.savez_compressed(pred_path, train=p_tr, validation=p_va, test=p_te, C=best_c, cv_auc=cv_auc)
        P_train[:, j], P_val[:, j], P_test[:, j] = p_tr, p_va, p_te
        selected_rows.append({"Endpoint": target, "C": best_c, "CV AUC": cv_auc})
        print(f"✅ {model_name} | {target} complete।")

    pd.DataFrame(selected_rows).to_csv(model_output / "selected_c.csv", index=False)
    result = register_model_result(model_name, P_train, P_val, P_test, notes="ECFP4 exact binary Tanimoto kernel; C selected by train-only CV.")
    return result, {r["Endpoint"]: r["C"] for r in selected_rows}

(svm_tanimoto_summary, svm_tanimoto_per_endpoint), TANIMOTO_C_BY_ENDPOINT = run_tanimoto_svm(
    "SVM Tanimoto", K_tr, K_va, K_te
)
save_runtime_session()


# Deep Learning Infrastructure

সব neural model-এর common rules:

- missing-label masking
- train-only class weighting যেখানে paper-aligned
- validation-based early stopping
- epoch-level checkpoint
- GPU থাকলে GPU, না থাকলে CPU
- completed model পুনরায় run করলে saved prediction load


In [ ]:
class TabularDataset(Dataset):
    def __init__(self, X, Y):
        self.X = torch.as_tensor(np.asarray(X), dtype=torch.float32)
        self.Y = torch.as_tensor(np.asarray(Y), dtype=torch.float32)
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx], self.Y[idx]

def compute_pos_weight(Ymat, cap=25.0):
    weights = []
    for j in range(Ymat.shape[1]):
        y = Ymat[:, j]
        pos = max(int(np.nansum(y == 1)), 1)
        neg = max(int(np.nansum(y == 0)), 1)
        weights.append(min(neg / pos, cap))
    return torch.tensor(weights, dtype=torch.float32, device=DEVICE)

def masked_weighted_bce(logits, targets, pos_weight=None):
    mask = torch.isfinite(targets)
    safe_targets = torch.nan_to_num(targets, nan=0.0)
    loss = F.binary_cross_entropy_with_logits(
        logits, safe_targets, reduction="none", pos_weight=pos_weight
    )
    return (loss * mask.float()).sum() / mask.float().sum().clamp_min(1.0)

@torch.no_grad()
def predict_tabular_torch(model, X, batch_size=512, probability_fn=None):
    model.eval()
    loader = DataLoader(torch.as_tensor(np.asarray(X), dtype=torch.float32), batch_size=batch_size, shuffle=False)
    out = []
    for xb in loader:
        raw = model(xb.to(DEVICE))
        prob = probability_fn(raw) if probability_fn is not None else torch.sigmoid(raw)
        out.append(prob.detach().cpu().numpy())
    return np.vstack(out)

def train_tabular_multitask(
    model_name, model, Xtr, Ytr, Xva, Yva, Xte, Yte,
    optimizer, max_epochs, batch_size, patience,
    loss_function=None, probability_fn=None, scheduler=None,
    notes=""
):
    model_dir = OUTPUT_DIR / slugify(model_name)
    checkpoint_dir = CHECKPOINT_DIR / slugify(model_name)
    model_save_dir = MODEL_DIR / slugify(model_name)
    model_dir.mkdir(parents=True, exist_ok=True)
    checkpoint_dir.mkdir(parents=True, exist_ok=True)
    model_save_dir.mkdir(parents=True, exist_ok=True)

    pred_path = model_dir / "predictions.npz"
    summary_path = model_dir / "overall_summary.json"
    if pred_path.exists() and summary_path.exists():
        print("♻️", model_name, "আগেই complete হয়েছে; saved result load হচ্ছে।")
        pred = np.load(pred_path)
        return register_model_result(model_name, pred["train"], pred["validation"], pred["test"], notes=notes)

    train_loader = DataLoader(
        TabularDataset(Xtr, Ytr), batch_size=batch_size, shuffle=True,
        num_workers=0, pin_memory=(DEVICE.type == "cuda")
    )
    val_loader = DataLoader(TabularDataset(Xva, Yva), batch_size=max(batch_size, 256), shuffle=False, num_workers=0)

    loss_function = loss_function or (lambda raw, y: masked_weighted_bce(raw, y, compute_pos_weight(Ytr)))
    checkpoint_path = checkpoint_dir / "last_checkpoint.pt"
    best_path = model_save_dir / "best_model.pt"
    history = {"train_loss": [], "val_loss": [], "val_auc": []}
    start_epoch, best_val_loss, wait = 0, float("inf"), 0

    if checkpoint_path.exists():
        ckpt = torch.load(checkpoint_path, map_location=DEVICE)
        model.load_state_dict(ckpt["model_state"])
        optimizer.load_state_dict(ckpt["optimizer_state"])
        if scheduler is not None and ckpt.get("scheduler_state") is not None:
            scheduler.load_state_dict(ckpt["scheduler_state"])
        start_epoch = int(ckpt["epoch"]) + 1
        best_val_loss = float(ckpt["best_val_loss"])
        wait = int(ckpt["wait"])
        history = ckpt["history"]
        print(f"♻️ {model_name}: epoch {start_epoch} থেকে training resume হবে।")

    model.to(DEVICE)
    for epoch in range(start_epoch, max_epochs):
        model.train()
        train_losses = []
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            raw = model(xb)
            loss = loss_function(raw, yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            optimizer.step()
            train_losses.append(float(loss.detach().cpu()))

        model.eval()
        val_losses = []
        val_probs, val_true = [], []
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                raw = model(xb)
                val_losses.append(float(loss_function(raw, yb).detach().cpu()))
                prob = probability_fn(raw) if probability_fn is not None else torch.sigmoid(raw)
                val_probs.append(prob.cpu().numpy())
                val_true.append(yb.cpu().numpy())

        tr_loss = float(np.mean(train_losses))
        va_loss = float(np.mean(val_losses))
        vp = np.vstack(val_probs)
        vy = np.vstack(val_true)
        va_auc = mean_metric([safe_auc(vy[:, j], vp[:, j]) for j in range(len(TARGETS))])
        history["train_loss"].append(tr_loss)
        history["val_loss"].append(va_loss)
        history["val_auc"].append(va_auc)

        if scheduler is not None:
            try:
                scheduler.step(va_loss)
            except TypeError:
                scheduler.step()

        improved = va_loss < best_val_loss - 1e-5
        if improved:
            best_val_loss = va_loss
            wait = 0
            torch.save(model.state_dict(), best_path)
        else:
            wait += 1

        torch.save({
            "epoch": epoch, "model_state": model.state_dict(),
            "optimizer_state": optimizer.state_dict(),
            "scheduler_state": scheduler.state_dict() if scheduler is not None else None,
            "best_val_loss": best_val_loss, "wait": wait, "history": history,
        }, checkpoint_path)

        print(f"Epoch {epoch+1:03d}/{max_epochs} | train_loss={tr_loss:.5f} | val_loss={va_loss:.5f} | val_auc={va_auc:.4f}")
        if wait >= patience:
            print(f"✅ Early stopping: {patience} epoch validation improvement হয়নি।")
            break

    if best_path.exists():
        model.load_state_dict(torch.load(best_path, map_location=DEVICE))
    model.to(DEVICE)

    P_train = predict_tabular_torch(model, Xtr, probability_fn=probability_fn)
    P_val = predict_tabular_torch(model, Xva, probability_fn=probability_fn)
    P_test = predict_tabular_torch(model, Xte, probability_fn=probability_fn)
    result = register_model_result(model_name, P_train, P_val, P_test, history=history, notes=notes)
    if checkpoint_path.exists():
        checkpoint_path.unlink()
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    return result


## Model 5 — DeepTox-style Multi-task DNN

Paper-aligned components:

- chemical cleaning, fragmentation/normalization concept
- high-dimensional fingerprints + descriptors
- median imputation and standardization
- `tanh`-scaled input
- ReLU hidden units
- 20% input dropout, 50% hidden dropout
- masked multi-task logistic loss
- mini-batch SGD
- L2 weight decay and early stopping

Open-source reproducible architecture: `2048 → 1024 → 512 → 12` (resource-safe adaptation of the paper's large architecture search space)।


In [ ]:
if "SOHOJ_READY" not in globals():
    import dill
    from pathlib import Path
    session_file = Path(Path("sohoj_session_path.txt").read_text(encoding="utf-8").strip())
    print("♻️ Kernel restart detected; saved session load হচ্ছে...")
    import __main__
    dill.load_module(str(session_file), module=__main__)
    if "rehydrate_runtime" in globals():
        rehydrate_runtime()

class DeepToxMTDNN(nn.Module):
    def __init__(self, input_dim, n_tasks):
        super().__init__()
        self.input_dropout = nn.Dropout(0.20)
        self.net = nn.Sequential(
            nn.Linear(input_dim, 2048), nn.ReLU(), nn.Dropout(0.50),
            nn.Linear(2048, 1024), nn.ReLU(), nn.Dropout(0.50),
            nn.Linear(1024, 512), nn.ReLU(), nn.Dropout(0.50),
            nn.Linear(512, n_tasks),
        )
    def forward(self, x):
        return self.net(self.input_dropout(x))

Xtr_deeptox = np.tanh(X_train_chem).astype(np.float32)
Xva_deeptox = np.tanh(X_val_chem).astype(np.float32)
Xte_deeptox = np.tanh(X_test_chem).astype(np.float32)

deeptox_model = DeepToxMTDNN(Xtr_deeptox.shape[1], len(TARGETS)).to(DEVICE)
pos_weight_deeptox = compute_pos_weight(Y_train, cap=20.0)
deeptox_optimizer = torch.optim.SGD(
    deeptox_model.parameters(), lr=0.01, momentum=0.9, weight_decay=1e-5
)
deeptox_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    deeptox_optimizer, mode="min", factor=0.5, patience=4, min_lr=1e-5
)

(deeptox_summary, deeptox_per_endpoint) = train_tabular_multitask(
    "DeepTox-style MT-DNN",
    deeptox_model,
    Xtr_deeptox, Y_train, Xva_deeptox, Y_val, Xte_deeptox, Y_test,
    optimizer=deeptox_optimizer,
    max_epochs=100 if RESOURCE_MODE != "high" else 160,
    batch_size=512,
    patience=12,
    loss_function=lambda raw, y: masked_weighted_bce(raw, y, pos_weight_deeptox),
    scheduler=deeptox_scheduler,
    notes="DeepTox paper-aligned open-source adaptation: ECFP4+MACCS+RDKit descriptors, tanh scaling, ReLU, 20% input dropout, 50% hidden dropout, masked weighted BCE, SGD."
)
save_runtime_session()


## Model 6 — Multitask CapsNet with RBM Layers and Dynamic Routing

Paper architecture closely reproduced:

- MACCS 166 bits + 13 molecular properties = 179 inputs
- Min–Max scaling to `[0, 1]` fit only on training data
- RBM1: 256 nodes
- RBM2: 128 nodes
- PrimaryCaps: 8 capsules × 8 dimensions
- task-specific DigitCaps: 2 class capsules × 2 dimensions
- 2 dynamic-routing iterations
- capsule margin loss
- batch size 148
- biased original training data retained (paper's key design)

`apparent logD at pH 7.4` এবং experimental solubility supplied CSV-তে নেই; RDKit LogP proxy এবং reproducible ESOL estimate ব্যবহার করা হয়েছে এবং code-এ স্পষ্টভাবে রাখা হয়েছে।


In [ ]:
if "SOHOJ_READY" not in globals():
    import dill
    from pathlib import Path
    session_file = Path(Path("sohoj_session_path.txt").read_text(encoding="utf-8").strip())
    print("♻️ Kernel restart detected; saved session load হচ্ছে...")
    import __main__
    dill.load_module(str(session_file), module=__main__)
    if "rehydrate_runtime" in globals():
        rehydrate_runtime()

def esol_estimate(mol):
    logp = Crippen.MolLogP(mol)
    mw = Descriptors.MolWt(mol)
    rot = Lipinski.NumRotatableBonds(mol)
    heavy = max(mol.GetNumHeavyAtoms(), 1)
    aromatic_atoms = sum(a.GetIsAromatic() for a in mol.GetAtoms())
    ap = aromatic_atoms / heavy
    return 0.16 - 1.5 * logp - 0.01 * (mw - 40.0) + 0.066 * rot + 0.066 * ap

def capsnet_13_descriptors(mol):
    logp = Crippen.MolLogP(mol)
    tpsa = rdMolDescriptors.CalcTPSA(mol)
    labute = rdMolDescriptors.CalcLabuteASA(mol)
    return [
        logp,                              # 1: octanol-water partition coefficient
        logp,                              # 2: reproducible proxy for apparent logD at pH 7.4
        esol_estimate(mol),                # 3: reproducible ESOL solubility estimate
        Descriptors.MolWt(mol),            # 4: molecular weight
        Lipinski.NumHDonors(mol),          # 5: H-bond donors
        Lipinski.NumHAcceptors(mol),       # 6: H-bond acceptors
        Lipinski.NumRotatableBonds(mol),   # 7: rotatable bonds
        rdMolDescriptors.CalcNumRings(mol),# 8: rings
        rdMolDescriptors.CalcNumAromaticRings(mol), # 9: aromatic rings
        sum(a.GetAtomicNum() in (7, 8) for a in mol.GetAtoms()), # 10: N + O atoms
        tpsa,                              # 11: polar surface area
        tpsa / max(labute, 1e-6),          # 12: fractional polar surface area
        labute,                            # 13: molecular surface area
    ]

CAPS_FEATURE_PATH = FEATURE_DIR / "capsnet_179.npy"
if CAPS_FEATURE_PATH.exists():
    X_caps_all = np.load(CAPS_FEATURE_PATH)
    print("♻️ CapsNet 179-feature cache load হয়েছে।")
else:
    rows = []
    for mol in tqdm(df["mol"], desc="CapsNet 179 features"):
        maccs = maccs_fp(mol)[1:]  # 166 informative MACCS bits
        rows.append(np.concatenate([maccs, np.asarray(capsnet_13_descriptors(mol), dtype=np.float32)]))
    X_caps_all = np.vstack(rows).astype(np.float32)
    np.save(CAPS_FEATURE_PATH, X_caps_all)

caps_scaler = MinMaxScaler(feature_range=(0, 1), clip=True)
Xtr_caps = caps_scaler.fit_transform(X_caps_all[train_idx]).astype(np.float32)
Xva_caps = caps_scaler.transform(X_caps_all[val_idx]).astype(np.float32)
Xte_caps = caps_scaler.transform(X_caps_all[test_idx]).astype(np.float32)
joblib.dump(caps_scaler, MODEL_DIR / "capsnet_minmax_scaler.joblib")
print("✅ CapsNet input:", Xtr_caps.shape, "| range:", float(Xtr_caps.min()), "to", float(Xtr_caps.max()))

save_runtime_session()


In [ ]:
if "SOHOJ_READY" not in globals():
    import dill
    from pathlib import Path
    session_file = Path(Path("sohoj_session_path.txt").read_text(encoding="utf-8").strip())
    print("♻️ Kernel restart detected; saved session load হচ্ছে...")
    import __main__
    dill.load_module(str(session_file), module=__main__)
    if "rehydrate_runtime" in globals():
        rehydrate_runtime()

RBM_DIR = MODEL_DIR / "multitask_capsnet_rbm"
RBM_DIR.mkdir(parents=True, exist_ok=True)
RBM1_PATH, RBM2_PATH = RBM_DIR / "rbm1.joblib", RBM_DIR / "rbm2.joblib"

if RBM1_PATH.exists() and RBM2_PATH.exists():
    rbm1 = joblib.load(RBM1_PATH)
    rbm2 = joblib.load(RBM2_PATH)
    print("♻️ দুইটি pretrained RBM cache থেকে load হয়েছে।")
else:
    print("⏳ RBM1 (179 → 256) pretraining শুরু...")
    rbm1 = BernoulliRBM(
        n_components=256, learning_rate=0.01, batch_size=148,
        n_iter=20 if RESOURCE_MODE == "low" else 30,
        verbose=1, random_state=SEED
    )
    H1 = rbm1.fit_transform(Xtr_caps)
    joblib.dump(rbm1, RBM1_PATH)

    print("⏳ RBM2 (256 → 128) pretraining শুরু...")
    rbm2 = BernoulliRBM(
        n_components=128, learning_rate=0.01, batch_size=148,
        n_iter=15 if RESOURCE_MODE == "low" else 25,
        verbose=1, random_state=SEED
    )
    rbm2.fit(H1)
    joblib.dump(rbm2, RBM2_PATH)
    print("✅ দুইটি RBM pretraining complete।")

save_runtime_session()


In [ ]:
if "SOHOJ_READY" not in globals():
    import dill
    from pathlib import Path
    session_file = Path(Path("sohoj_session_path.txt").read_text(encoding="utf-8").strip())
    print("♻️ Kernel restart detected; saved session load হচ্ছে...")
    import __main__
    dill.load_module(str(session_file), module=__main__)
    if "rehydrate_runtime" in globals():
        rehydrate_runtime()

def squash_capsules(s, dim=-1, eps=1e-8):
    norm_sq = (s * s).sum(dim=dim, keepdim=True)
    scale = norm_sq / (1.0 + norm_sq) / torch.sqrt(norm_sq + eps)
    return scale * s

class MultitaskCapsNetRBM(nn.Module):
    def __init__(self, input_dim=179, n_tasks=12, routing_iters=2):
        super().__init__()
        self.n_tasks = n_tasks
        self.routing_iters = routing_iters
        self.dropout = nn.Dropout(0.20)
        self.rbm1 = nn.Linear(input_dim, 256)
        self.rbm2 = nn.Linear(256, 128)
        self.primary = nn.Linear(128, 8 * 8)
        self.route_W = nn.Parameter(0.02 * torch.randn(n_tasks, 8, 2, 2, 8))

    def forward(self, x):
        x = torch.sigmoid(self.rbm1(self.dropout(x)))
        x = torch.sigmoid(self.rbm2(self.dropout(x)))
        primary = squash_capsules(self.primary(x).view(-1, 8, 8), dim=-1)

        # B,T,N,O,D = batch, tasks, primary capsules, class capsules, capsule dim
        u_hat = torch.einsum("bni,tnodi->btnod", primary, self.route_W)
        b = torch.zeros(u_hat.shape[:-1], device=x.device, dtype=x.dtype)
        for r in range(self.routing_iters):
            c = torch.softmax(b, dim=-1)
            s = (c.unsqueeze(-1) * u_hat).sum(dim=2)
            v = squash_capsules(s, dim=-1)
            if r < self.routing_iters - 1:
                b = b + (u_hat * v.unsqueeze(2)).sum(dim=-1)
        return torch.linalg.vector_norm(v, dim=-1)  # [B, tasks, 2]

def capsule_margin_loss(lengths, targets, m_plus=0.9, m_minus=0.1, lam=0.5):
    mask = torch.isfinite(targets)
    safe = torch.nan_to_num(targets, nan=0.0).long()
    one_hot = F.one_hot(safe, num_classes=2).float()
    left = F.relu(m_plus - lengths).pow(2)
    right = F.relu(lengths - m_minus).pow(2)
    loss = one_hot * left + lam * (1.0 - one_hot) * right
    loss = loss.sum(dim=-1)
    return (loss * mask.float()).sum() / mask.float().sum().clamp_min(1.0)

def capsule_probability(lengths):
    return lengths[..., 1] / lengths.sum(dim=-1).clamp_min(1e-8)

caps_model = MultitaskCapsNetRBM(n_tasks=len(TARGETS), routing_iters=2).to(DEVICE)
with torch.no_grad():
    caps_model.rbm1.weight.copy_(torch.tensor(rbm1.components_, dtype=torch.float32, device=DEVICE))
    caps_model.rbm1.bias.copy_(torch.tensor(rbm1.intercept_hidden_, dtype=torch.float32, device=DEVICE))
    caps_model.rbm2.weight.copy_(torch.tensor(rbm2.components_, dtype=torch.float32, device=DEVICE))
    caps_model.rbm2.bias.copy_(torch.tensor(rbm2.intercept_hidden_, dtype=torch.float32, device=DEVICE))

caps_optimizer = torch.optim.Adam(caps_model.parameters(), lr=1e-3)
caps_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(caps_optimizer, mode="min", factor=0.5, patience=8, min_lr=1e-6)

CAPS_EPOCHS = 300 if RESOURCE_MODE == "low" else (500 if RESOURCE_MODE == "standard" else 1000)
(caps_summary, caps_per_endpoint) = train_tabular_multitask(
    "Multitask CapsNet + RBM + Dynamic Routing",
    caps_model,
    Xtr_caps, Y_train, Xva_caps, Y_val, Xte_caps, Y_test,
    optimizer=caps_optimizer,
    max_epochs=CAPS_EPOCHS,
    batch_size=148,
    patience=40,
    loss_function=capsule_margin_loss,
    probability_fn=capsule_probability,
    scheduler=caps_scheduler,
    notes="Paper-aligned 179 inputs, two RBMs (256/128), PrimaryCaps 8x8, task-specific 2x2 DigitCaps, two routing iterations, margin loss. LogD and solubility use documented open-source proxies."
)
save_runtime_session()


## Model 7 — DenseNet121 + SVM Tanimoto

এই hybrid model supplied papers-এ সরাসরি specified ছিল না; এটি user-requested hybrid। Pipeline:

1. standardized molecule → 2D structure image
2. ImageNet-pretrained DenseNet121 → 1024-dimensional visual embedding
3. train-only PCA + Min–Max scaling
4. ECFP4 Tanimoto kernel + continuous visual Tanimoto kernel
5. weighted composite kernel: `0.70 × chemical + 0.30 × visual`
6. endpoint-wise SVM; SVM Tanimoto model-এর train-only selected `C` reuse

DenseNet feature extraction batch-level cache ব্যবহার করে; interruption হলে next run completed rows থেকে resume করবে।


In [ ]:
if "SOHOJ_READY" not in globals():
    import dill
    from pathlib import Path
    session_file = Path(Path("sohoj_session_path.txt").read_text(encoding="utf-8").strip())
    print("♻️ Kernel restart detected; saved session load হচ্ছে...")
    import __main__
    dill.load_module(str(session_file), module=__main__)
    if "rehydrate_runtime" in globals():
        rehydrate_runtime()

if not TORCHVISION_AVAILABLE:
    print("⚠️ torchvision unavailable; DenseNet121 model run করা যাবে না। Setup cell পুনরায় run করুন।")
else:
    DENSE_DIR = CACHE_DIR / f"densenet121_{RUN_ID}"
    DENSE_DIR.mkdir(parents=True, exist_ok=True)
    DENSE_PATH = DENSE_DIR / "features_1024.npy"
    DENSE_PROGRESS = DENSE_DIR / "progress.json"

    class MoleculeImageDataset(Dataset):
        def __init__(self, smiles_list, indices, transform):
            self.smiles_list = smiles_list
            self.indices = list(indices)
            self.transform = transform
        def __len__(self):
            return len(self.indices)
        def __getitem__(self, k):
            idx = self.indices[k]
            mol = Chem.MolFromSmiles(self.smiles_list[idx])
            img = Draw.MolToImage(mol, size=(224, 224)).convert("RGB")
            return self.transform(img), idx

    try:
        weights = DenseNet121_Weights.DEFAULT
        dense_transform = weights.transforms()
        dense_backbone = densenet121(weights=weights)
        DENSENET_PRETRAINED = True
        print("✅ ImageNet-pretrained DenseNet121 load হয়েছে।")
    except Exception as exc:
        print("⚠️ Pretrained weights download/load হয়নি:", exc)
        print("ℹ️ Code continuity-এর জন্য random weights ব্যবহার হবে; better result-এর জন্য internet সহ cell rerun করুন।")
        dense_transform = transforms.Compose([
            transforms.Resize((224, 224)), transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
        ])
        dense_backbone = densenet121(weights=None)
        DENSENET_PRETRAINED = False

    dense_backbone.classifier = nn.Identity()
    dense_backbone = dense_backbone.to(DEVICE).eval()

    if DENSE_PATH.exists() and DENSE_PROGRESS.exists() and json.loads(DENSE_PROGRESS.read_text()).get("done"):
        X_dense_all = np.load(DENSE_PATH, mmap_mode="r")
        print("♻️ DenseNet121 feature cache সম্পূর্ণ; load হয়েছে।")
    else:
        if not DENSE_PATH.exists():
            dense_mem = np.lib.format.open_memmap(DENSE_PATH, mode="w+", dtype="float32", shape=(len(df), 1024))
            completed = 0
        else:
            dense_mem = np.lib.format.open_memmap(DENSE_PATH, mode="r+", dtype="float32", shape=(len(df), 1024))
            completed = int(json.loads(DENSE_PROGRESS.read_text()).get("completed", 0)) if DENSE_PROGRESS.exists() else 0

        remaining = np.arange(completed, len(df))
        ds = MoleculeImageDataset(df["canonical_smiles"].tolist(), remaining, dense_transform)
        loader = DataLoader(ds, batch_size=BATCH_IMAGE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=(DEVICE.type == "cuda"))

        with torch.no_grad():
            for xb, idxb in tqdm(loader, desc="DenseNet121 feature extraction"):
                feat = dense_backbone(xb.to(DEVICE)).detach().cpu().numpy().astype(np.float32)
                idx_np = np.asarray(idxb, dtype=int)
                dense_mem[idx_np] = feat
                dense_mem.flush()
                completed = int(idx_np.max()) + 1
                DENSE_PROGRESS.write_text(json.dumps({"completed": completed, "done": False}), encoding="utf-8")

        DENSE_PROGRESS.write_text(json.dumps({"completed": len(df), "done": True, "pretrained": DENSENET_PRETRAINED}), encoding="utf-8")
        X_dense_all = np.load(DENSE_PATH, mmap_mode="r")
        print("✅ DenseNet121 feature extraction complete।")

    print("DenseNet feature shape:", X_dense_all.shape)

save_runtime_session()


In [ ]:
if "SOHOJ_READY" not in globals():
    import dill
    from pathlib import Path
    session_file = Path(Path("sohoj_session_path.txt").read_text(encoding="utf-8").strip())
    print("♻️ Kernel restart detected; saved session load হচ্ছে...")
    import __main__
    dill.load_module(str(session_file), module=__main__)
    if "rehydrate_runtime" in globals():
        rehydrate_runtime()

if TORCHVISION_AVAILABLE and "X_dense_all" in globals():
    DENSE_KERNEL_DIR = CACHE_DIR / f"densenet_tanimoto_kernels_{RUN_ID}"
    DENSE_KERNEL_DIR.mkdir(parents=True, exist_ok=True)
    PCA_PATH = MODEL_DIR / "densenet_pca_minmax.joblib"

    if PCA_PATH.exists():
        dense_transformer = joblib.load(PCA_PATH)
        dense_pca, dense_minmax = dense_transformer["pca"], dense_transformer["minmax"]
    else:
        pca_components = min(128, len(train_idx) - 1, X_dense_all.shape[1])
        dense_pca = PCA(n_components=pca_components, random_state=SEED, svd_solver="randomized")
        ztr = dense_pca.fit_transform(np.asarray(X_dense_all[train_idx]))
        dense_minmax = MinMaxScaler(feature_range=(0, 1), clip=True).fit(ztr)
        joblib.dump({"pca": dense_pca, "minmax": dense_minmax}, PCA_PATH)

    Ztr = dense_minmax.transform(dense_pca.transform(np.asarray(X_dense_all[train_idx]))).astype(np.float32)
    Zva = dense_minmax.transform(dense_pca.transform(np.asarray(X_dense_all[val_idx]))).astype(np.float32)
    Zte = dense_minmax.transform(dense_pca.transform(np.asarray(X_dense_all[test_idx]))).astype(np.float32)

    def compute_continuous_tanimoto_cached(A, B, path, block_size=256):
        path = Path(path)
        if path.exists():
            return np.load(path, mmap_mode="r")
        A, B = np.asarray(A, np.float32), np.asarray(B, np.float32)
        out = np.lib.format.open_memmap(path, mode="w+", dtype="float32", shape=(len(A), len(B)))
        b2 = (B * B).sum(axis=1)[None, :]
        for start in tqdm(range(0, len(A), block_size), desc=f"Visual Tanimoto {path.stem}"):
            end = min(start + block_size, len(A))
            block = A[start:end]
            dot = block @ B.T
            denom = (block * block).sum(axis=1)[:, None] + b2 - dot
            out[start:end] = np.divide(dot, denom, out=np.zeros_like(dot), where=denom > 1e-8)
            out.flush()
        return np.load(path, mmap_mode="r")

    Kv_tr = compute_continuous_tanimoto_cached(Ztr, Ztr, DENSE_KERNEL_DIR / "visual_train.npy")
    Kv_va = compute_continuous_tanimoto_cached(Zva, Ztr, DENSE_KERNEL_DIR / "visual_validation.npy")
    Kv_te = compute_continuous_tanimoto_cached(Zte, Ztr, DENSE_KERNEL_DIR / "visual_test.npy")

    Khy_tr_path = DENSE_KERNEL_DIR / "hybrid_train.npy"
    Khy_va_path = DENSE_KERNEL_DIR / "hybrid_validation.npy"
    Khy_te_path = DENSE_KERNEL_DIR / "hybrid_test.npy"
    for path, kc, kv in [(Khy_tr_path, K_tr, Kv_tr), (Khy_va_path, K_va, Kv_va), (Khy_te_path, K_te, Kv_te)]:
        if not path.exists():
            out = np.lib.format.open_memmap(path, mode="w+", dtype="float32", shape=kc.shape)
            out[:] = 0.70 * np.asarray(kc) + 0.30 * np.asarray(kv)
            out.flush()

    Khy_tr = np.load(Khy_tr_path, mmap_mode="r")
    Khy_va = np.load(Khy_va_path, mmap_mode="r")
    Khy_te = np.load(Khy_te_path, mmap_mode="r")

    (dense_svm_summary, dense_svm_per_endpoint), _ = run_tanimoto_svm(
        "DenseNet121 + SVM Tanimoto", Khy_tr, Khy_va, Khy_te,
        reuse_c=TANIMOTO_C_BY_ENDPOINT
    )
else:
    print("⚠️ DenseNet121 + SVM Tanimoto skipped; DenseNet feature unavailable।")

save_runtime_session()


## Model 8 — Multi-channel 2D CNN

2019 multi-channel CNN paper-এর key preprocessing:

- 24 Å × 24 Å grid
- 0.5 Å resolution → 48 × 48 pixels
- six channels: hydrogen bond, hydrophobicity, metallicity, excluded volume, positive ionization, negative ionization
- Van der Waals-style spatial contribution
- hydrogen-bond 12–10 potential-style contribution
- training-only augmentation, batch normalization and dropout
- four-stage CNN
- 60 epochs maximum

**Reproducibility note:** Paper-এর exact ChemAxon Standardizer/AutoDock parameters supplied নয়। RDKit atom typing এবং documented numerical approximation ব্যবহার করা হয়েছে। Raw-grid SMOTE chemically questionable এবং multi-task missing-label setting-এ ambiguous; তাই paper-এর positive weighting-এর সাথে task-aware oversampling এবং geometric image augmentation ব্যবহার করা হয়েছে।


In [ ]:
if "SOHOJ_READY" not in globals():
    import dill
    from pathlib import Path
    session_file = Path(Path("sohoj_session_path.txt").read_text(encoding="utf-8").strip())
    print("♻️ Kernel restart detected; saved session load হচ্ছে...")
    import __main__
    dill.load_module(str(session_file), module=__main__)
    if "rehydrate_runtime" in globals():
        rehydrate_runtime()

GRID_SIZE = 48
GRID_HALF = 12.0
GRID_AXIS = np.linspace(-GRID_HALF, GRID_HALF, GRID_SIZE, dtype=np.float32)
GX, GY = np.meshgrid(GRID_AXIS, GRID_AXIS, indexing="xy")
PTABLE = Chem.GetPeriodicTable()
FEATURE_FACTORY = ChemicalFeatures.BuildFeatureFactory(str(Path(RDConfig.RDDataDir) / "BaseFeatures.fdef"))
METALS = {3, 4, 11, 12, 13, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 37, 38, 47, 48, 55, 56, 78, 79, 80}
HYDROPHOBIC = {6, 9, 16, 17, 35, 53}

def molecule_six_channel_grid(mol):
    mol = Chem.Mol(mol)
    AllChem.Compute2DCoords(mol)
    conf = mol.GetConformer()
    coords = np.array([[conf.GetAtomPosition(i).x, conf.GetAtomPosition(i).y] for i in range(mol.GetNumAtoms())], dtype=np.float32)
    coords -= coords.mean(axis=0, keepdims=True)
    span = max(np.ptp(coords[:, 0]), np.ptp(coords[:, 1]), 1.0)
    coords *= min(1.0, 20.0 / span)

    donor_ids, acceptor_ids = set(), set()
    for feat in FEATURE_FACTORY.GetFeaturesForMol(mol):
        if feat.GetFamily() == "Donor": donor_ids.update(feat.GetAtomIds())
        if feat.GetFamily() == "Acceptor": acceptor_ids.update(feat.GetAtomIds())

    channels = np.zeros((6, GRID_SIZE, GRID_SIZE), dtype=np.float32)
    for i, atom in enumerate(mol.GetAtoms()):
        x, y = coords[i]
        r = np.sqrt((GX - x) ** 2 + (GY - y) ** 2) + 0.25
        z = atom.GetAtomicNum()
        vdw = float(PTABLE.GetRvdw(z) or 1.7)
        vdw_field = 1.0 - np.exp(-np.clip((vdw / r) ** 12, 0, 50))

        # 1. Hydrogen-bond donor/acceptor channel: stable 12-10 potential approximation.
        if i in donor_ids or i in acceptor_ids:
            c, d = (1.0, 1.2) if z in (7, 8) else (0.8, 1.0)
            hfield = np.abs(c / np.clip(r, 0.65, None) ** 12 - d / np.clip(r, 0.65, None) ** 10)
            channels[0] += np.clip(hfield, 0, 10)

        # 2. Hydrophobicity
        if z in HYDROPHOBIC:
            channels[1] += vdw_field
        # 3. Metallicity
        if z in METALS:
            channels[2] += vdw_field
        # 4. Excluded volume — all atoms
        channels[3] += vdw_field
        # 5. Positive ionization
        if atom.GetFormalCharge() > 0 or (z in (7, 15) and atom.GetTotalDegree() >= 4):
            channels[4] += vdw_field
        # 6. Negative ionization
        if atom.GetFormalCharge() < 0 or (z in (8, 16) and atom.GetTotalDegree() == 1):
            channels[5] += vdw_field

    for c in range(6):
        vmax = float(np.percentile(channels[c], 99.5))
        if vmax > 0:
            channels[c] = np.clip(channels[c] / vmax, 0, 1)
    return channels.astype(np.float16)

GRID_DIR = CACHE_DIR / f"six_channel_grids_{RUN_ID}"
GRID_DIR.mkdir(parents=True, exist_ok=True)
GRID_PATH = GRID_DIR / "grids_float16.npy"
GRID_PROGRESS = GRID_DIR / "progress.json"

if GRID_PATH.exists() and GRID_PROGRESS.exists() and json.loads(GRID_PROGRESS.read_text()).get("done"):
    X_grids = np.load(GRID_PATH, mmap_mode="r")
    print("♻️ Six-channel grid cache সম্পূর্ণ; load হয়েছে।")
else:
    if not GRID_PATH.exists():
        grid_mem = np.lib.format.open_memmap(GRID_PATH, mode="w+", dtype="float16", shape=(len(df), 6, GRID_SIZE, GRID_SIZE))
        completed = 0
    else:
        grid_mem = np.lib.format.open_memmap(GRID_PATH, mode="r+", dtype="float16", shape=(len(df), 6, GRID_SIZE, GRID_SIZE))
        completed = int(json.loads(GRID_PROGRESS.read_text()).get("completed", 0)) if GRID_PROGRESS.exists() else 0

    for i in tqdm(range(completed, len(df)), desc="Six-channel molecular grid"):
        grid_mem[i] = molecule_six_channel_grid(df.iloc[i]["mol"])
        if (i + 1) % 25 == 0 or i + 1 == len(df):
            grid_mem.flush()
            GRID_PROGRESS.write_text(json.dumps({"completed": i + 1, "done": False}), encoding="utf-8")
    GRID_PROGRESS.write_text(json.dumps({"completed": len(df), "done": True}), encoding="utf-8")
    X_grids = np.load(GRID_PATH, mmap_mode="r")
    print("✅ Six-channel molecular grid generation complete।")

print("Grid tensor shape:", X_grids.shape)

save_runtime_session()


In [ ]:
if "SOHOJ_READY" not in globals():
    import dill
    from pathlib import Path
    session_file = Path(Path("sohoj_session_path.txt").read_text(encoding="utf-8").strip())
    print("♻️ Kernel restart detected; saved session load হচ্ছে...")
    import __main__
    dill.load_module(str(session_file), module=__main__)
    if "rehydrate_runtime" in globals():
        rehydrate_runtime()

fig, axes = plt.subplots(2, 3, figsize=(12, 8))
channel_names = ["Hydrogen Bond", "Hydrophobicity", "Metallicity", "Excluded Volume", "Positive Ionization", "Negative Ionization"]
sample_grid = np.asarray(X_grids[0], dtype=np.float32)
for ax, grid, name in zip(axes.flat, sample_grid, channel_names):
    im = ax.imshow(grid, cmap="magma", origin="lower")
    ax.set_title(name, fontweight="bold")
    ax.axis("off")
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.suptitle("Multi-channel 2D Molecular Grid — Sample Compound", fontweight="bold", fontsize=15)
plt.tight_layout()
plt.savefig(FIGURE_DIR / "03_multichannel_grid_sample.png", dpi=220, bbox_inches="tight")
plt.show()


In [ ]:
if "SOHOJ_READY" not in globals():
    import dill
    from pathlib import Path
    session_file = Path(Path("sohoj_session_path.txt").read_text(encoding="utf-8").strip())
    print("♻️ Kernel restart detected; saved session load হচ্ছে...")
    import __main__
    dill.load_module(str(session_file), module=__main__)
    if "rehydrate_runtime" in globals():
        rehydrate_runtime()

class GridDataset(Dataset):
    def __init__(self, grid_array, indices, labels, augment=False):
        self.grid_array = grid_array
        self.indices = np.asarray(indices, dtype=int)
        self.labels = np.asarray(labels, dtype=np.float32)
        self.augment = augment
    def __len__(self):
        return len(self.indices)
    def __getitem__(self, k):
        x = np.asarray(self.grid_array[self.indices[k]], dtype=np.float32).copy()
        if self.augment:
            rot = np.random.randint(0, 4)
            x = np.rot90(x, k=rot, axes=(1, 2)).copy()
            if np.random.rand() < 0.5:
                x = x[:, :, ::-1].copy()
        return torch.from_numpy(x), torch.from_numpy(self.labels[k])

class MultiChannel2DCNN(nn.Module):
    def __init__(self, n_tasks):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(6, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(128, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(), nn.AdaptiveAvgPool2d((1, 1)),
        )
        self.head = nn.Sequential(nn.Flatten(), nn.Dropout(0.30), nn.Linear(256, 128), nn.ReLU(), nn.Dropout(0.30), nn.Linear(128, n_tasks))
    def forward(self, x):
        return self.head(self.features(x))

def train_grid_cnn():
    model_name = "Multi-channel 2D CNN"
    model_dir = OUTPUT_DIR / slugify(model_name)
    ckpt_dir = CHECKPOINT_DIR / slugify(model_name)
    save_dir = MODEL_DIR / slugify(model_name)
    for d in [model_dir, ckpt_dir, save_dir]: d.mkdir(parents=True, exist_ok=True)

    pred_path = model_dir / "predictions.npz"
    if pred_path.exists() and (model_dir / "overall_summary.json").exists():
        pred = np.load(pred_path)
        return register_model_result(model_name, pred["train"], pred["validation"], pred["test"], notes="Paper-aligned open-source six-channel 48x48 grid CNN.")

    # Task-aware sample weights: compounds with rare positive labels are sampled more often.
    endpoint_pos = np.nansum(Y_train == 1, axis=0)
    endpoint_neg = np.nansum(Y_train == 0, axis=0)
    rarity = endpoint_neg / np.maximum(endpoint_pos, 1)
    sample_weights = np.ones(len(train_idx), dtype=np.float64)
    for i in range(len(train_idx)):
        positive_tasks = np.where(Y_train[i] == 1)[0]
        if len(positive_tasks):
            sample_weights[i] += float(np.mean(np.minimum(rarity[positive_tasks], 20.0)))
    sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)

    train_ds = GridDataset(X_grids, train_idx, Y_train, augment=True)
    val_ds = GridDataset(X_grids, val_idx, Y_val, augment=False)
    test_ds = GridDataset(X_grids, test_idx, Y_test, augment=False)
    train_loader = DataLoader(train_ds, batch_size=BATCH_IMAGE, sampler=sampler, num_workers=NUM_WORKERS, pin_memory=(DEVICE.type == "cuda"))
    val_loader = DataLoader(val_ds, batch_size=max(BATCH_IMAGE, 32), shuffle=False, num_workers=NUM_WORKERS)

    model = MultiChannel2DCNN(len(TARGETS)).to(DEVICE)
    pos_weight = compute_pos_weight(Y_train, cap=25.0)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-3)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=4, min_lr=1e-6)
    ckpt_path, best_path = ckpt_dir / "last.pt", save_dir / "best_model.pt"
    history = {"train_loss": [], "val_loss": [], "val_auc": []}
    start_epoch, best_loss, wait = 0, float("inf"), 0

    if ckpt_path.exists():
        ckpt = torch.load(ckpt_path, map_location=DEVICE)
        model.load_state_dict(ckpt["model_state"]); optimizer.load_state_dict(ckpt["optimizer_state"])
        scheduler.load_state_dict(ckpt["scheduler_state"])
        start_epoch, best_loss, wait, history = ckpt["epoch"] + 1, ckpt["best_loss"], ckpt["wait"], ckpt["history"]
        print(f"♻️ Multi-channel CNN epoch {start_epoch} থেকে resume হবে।")

    max_epochs, patience = 60, 10
    for epoch in range(start_epoch, max_epochs):
        model.train(); train_losses=[]
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            logits = model(xb)
            loss = masked_weighted_bce(logits, yb, pos_weight)
            loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0); optimizer.step()
            train_losses.append(float(loss.detach().cpu()))

        model.eval(); val_losses=[]; probs=[]; trues=[]
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                logits = model(xb)
                val_losses.append(float(masked_weighted_bce(logits, yb, pos_weight).cpu()))
                probs.append(torch.sigmoid(logits).cpu().numpy()); trues.append(yb.cpu().numpy())
        tr_loss, va_loss = float(np.mean(train_losses)), float(np.mean(val_losses))
        vp, vy = np.vstack(probs), np.vstack(trues)
        va_auc = mean_metric([safe_auc(vy[:, j], vp[:, j]) for j in range(len(TARGETS))])
        history["train_loss"].append(tr_loss); history["val_loss"].append(va_loss); history["val_auc"].append(va_auc)
        scheduler.step(va_loss)

        if va_loss < best_loss - 1e-5:
            best_loss, wait = va_loss, 0
            torch.save(model.state_dict(), best_path)
        else:
            wait += 1
        torch.save({"epoch": epoch, "model_state": model.state_dict(), "optimizer_state": optimizer.state_dict(), "scheduler_state": scheduler.state_dict(), "best_loss": best_loss, "wait": wait, "history": history}, ckpt_path)
        print(f"Epoch {epoch+1:02d}/60 | train_loss={tr_loss:.5f} | val_loss={va_loss:.5f} | val_auc={va_auc:.4f}")
        if wait >= patience:
            print("✅ Early stopping activated।"); break

    if best_path.exists(): model.load_state_dict(torch.load(best_path, map_location=DEVICE))

    @torch.no_grad()
    def predict_grid(indices, labels):
        ds = GridDataset(X_grids, indices, labels, augment=False)
        loader = DataLoader(ds, batch_size=max(BATCH_IMAGE, 32), shuffle=False, num_workers=NUM_WORKERS)
        model.eval(); out=[]
        for xb, _ in loader:
            out.append(torch.sigmoid(model(xb.to(DEVICE))).cpu().numpy())
        return np.vstack(out)

    P_train = predict_grid(train_idx, Y_train)
    P_val = predict_grid(val_idx, Y_val)
    P_test = predict_grid(test_idx, Y_test)
    result = register_model_result(
        model_name, P_train, P_val, P_test, history=history,
        notes="Paper-aligned open-source six-channel 24A/0.5A (48x48) CNN with RDKit atom typing, task-aware oversampling, weighted masked BCE, BN, dropout and geometric augmentation."
    )
    if ckpt_path.exists(): ckpt_path.unlink()
    return result

cnn_summary, cnn_per_endpoint = train_grid_cnn()
save_runtime_session()


## Model 9 — Compact MTL-DNN

Supplied MTL-DNN paper setup:

- ECFP4 2048 bits + 7 RDKit descriptors = 2055 inputs
- hidden layers: `1024 → 512 → 128`
- ReLU
- Dropout 0.30
- 12 simultaneous sigmoid outputs
- class-weighted masked BCE
- Adam, learning rate `1e-4`
- early stopping

Paper reports 5-fold CV; user-requested fixed 70/10/20 split is used for final evaluation. RF/ET/XGB-এর feature representation এতে পরিবর্তন করা হয়নি।


In [ ]:
if "SOHOJ_READY" not in globals():
    import dill
    from pathlib import Path
    session_file = Path(Path("sohoj_session_path.txt").read_text(encoding="utf-8").strip())
    print("♻️ Kernel restart detected; saved session load হচ্ছে...")
    import __main__
    dill.load_module(str(session_file), module=__main__)
    if "rehydrate_runtime" in globals():
        rehydrate_runtime()

MTL_DESC_FUNCS = [
    ("MolWt", Descriptors.MolWt),
    ("MolLogP", Crippen.MolLogP),
    ("HBD", Lipinski.NumHDonors),
    ("HBA", Lipinski.NumHAcceptors),
    ("TPSA", rdMolDescriptors.CalcTPSA),
    ("RotB", Lipinski.NumRotatableBonds),
    ("RingCount", rdMolDescriptors.CalcNumRings),
]

MTL_DESC_PATH = FEATURE_DIR / "mtl_7_descriptors.npy"
if MTL_DESC_PATH.exists():
    X_mtl_desc = np.load(MTL_DESC_PATH)
else:
    X_mtl_desc = np.asarray([[float(fn(m)) for _, fn in MTL_DESC_FUNCS] for m in tqdm(df["mol"], desc="MTL 7 descriptors")], dtype=np.float32)
    np.save(MTL_DESC_PATH, X_mtl_desc)

mtl_imp = SimpleImputer(strategy="median")
mtl_scaler = StandardScaler()
Dtr = mtl_scaler.fit_transform(mtl_imp.fit_transform(X_mtl_desc[train_idx]))
Dva = mtl_scaler.transform(mtl_imp.transform(X_mtl_desc[val_idx]))
Dte = mtl_scaler.transform(mtl_imp.transform(X_mtl_desc[test_idx]))
Xtr_mtl = np.hstack([np.asarray(X_morgan[train_idx]), Dtr]).astype(np.float32)
Xva_mtl = np.hstack([np.asarray(X_morgan[val_idx]), Dva]).astype(np.float32)
Xte_mtl = np.hstack([np.asarray(X_morgan[test_idx]), Dte]).astype(np.float32)
joblib.dump({"imputer": mtl_imp, "scaler": mtl_scaler, "descriptor_names": [x[0] for x in MTL_DESC_FUNCS]}, MODEL_DIR / "mtl_dnn_preprocessing.joblib")

class CompactMTLDNN(nn.Module):
    def __init__(self, input_dim, n_tasks):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 1024), nn.ReLU(), nn.Dropout(0.30),
            nn.Linear(1024, 512), nn.ReLU(), nn.Dropout(0.30),
            nn.Linear(512, 128), nn.ReLU(), nn.Dropout(0.30),
            nn.Linear(128, n_tasks),
        )
    def forward(self, x):
        return self.net(x)

mtl_model = CompactMTLDNN(2055, len(TARGETS)).to(DEVICE)
mtl_pos_weight = compute_pos_weight(Y_train, cap=25.0)
mtl_optimizer = torch.optim.Adam(mtl_model.parameters(), lr=1e-4)
mtl_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(mtl_optimizer, mode="min", factor=0.5, patience=5, min_lr=1e-6)

(mtl_summary, mtl_per_endpoint) = train_tabular_multitask(
    "Compact MTL-DNN",
    mtl_model,
    Xtr_mtl, Y_train, Xva_mtl, Y_val, Xte_mtl, Y_test,
    optimizer=mtl_optimizer,
    max_epochs=150,
    batch_size=BATCH_TABULAR,
    patience=15,
    loss_function=lambda raw, y: masked_weighted_bce(raw, y, mtl_pos_weight),
    scheduler=mtl_scheduler,
    notes="Paper-aligned 2055 inputs, 1024-512-128 ReLU layers, dropout 0.3, Adam lr=1e-4, class-weighted masked BCE and early stopping."
)
save_runtime_session()


## Model 10 — Validation-weighted Soft-Voting Ensemble

- test labels কোনো weight selection-এ ব্যবহার হবে না
- endpoint-wise validation AUC থেকে weight নির্ধারণ
- weak model-এর weight shrink করা হবে
- missing/failed model automatically বাদ যাবে
- RF/Extra Trees bagging diversity + XGBoost boosting + kernel/deep/image models combine করা হবে


In [ ]:
if "SOHOJ_READY" not in globals():
    import dill
    from pathlib import Path
    session_file = Path(Path("sohoj_session_path.txt").read_text(encoding="utf-8").strip())
    print("♻️ Kernel restart detected; saved session load হচ্ছে...")
    import __main__
    dill.load_module(str(session_file), module=__main__)
    if "rehydrate_runtime" in globals():
        rehydrate_runtime()

def completed_prediction_sets():
    items = []
    for summary_path in OUTPUT_DIR.glob("*/overall_summary.json"):
        model_dir = summary_path.parent
        pred_path = model_dir / "predictions.npz"
        if not pred_path.exists():
            continue
        summary = json.loads(summary_path.read_text(encoding="utf-8"))
        if summary.get("Model") == "Validation-weighted Soft-Voting Ensemble":
            continue
        pred = np.load(pred_path)
        items.append((summary["Model"], np.asarray(pred["train"]), np.asarray(pred["validation"]), np.asarray(pred["test"])))
    return items

base_predictions = completed_prediction_sets()
if len(base_predictions) < 2:
    raise RuntimeError("Ensemble-এর জন্য অন্তত 2টি completed base model দরকার। আগে model cells run করুন।")

model_names = [x[0] for x in base_predictions]
weights = np.zeros((len(model_names), len(TARGETS)), dtype=float)
for m, (_, _, Pva, _) in enumerate(base_predictions):
    for j in range(len(TARGETS)):
        auc = safe_auc(Y_val[:, j], Pva[:, j])
        # Shrunk validation weight; 0.5-এর নিচে model-ও tiny positive weight পায়.
        weights[m, j] = max((auc - 0.5) ** 2, 0.0025) if np.isfinite(auc) else 0.0025
weights /= np.maximum(weights.sum(axis=0, keepdims=True), 1e-12)

P_train_ens = np.zeros_like(base_predictions[0][1], dtype=float)
P_val_ens = np.zeros_like(base_predictions[0][2], dtype=float)
P_test_ens = np.zeros_like(base_predictions[0][3], dtype=float)
for m, (_, Ptr, Pva, Pte) in enumerate(base_predictions):
    P_train_ens += Ptr * weights[m][None, :]
    P_val_ens += Pva * weights[m][None, :]
    P_test_ens += Pte * weights[m][None, :]

weight_df = pd.DataFrame(weights, index=model_names, columns=TARGETS)
ensemble_dir = OUTPUT_DIR / slugify("Validation-weighted Soft-Voting Ensemble")
ensemble_dir.mkdir(parents=True, exist_ok=True)
weight_df.to_csv(ensemble_dir / "validation_auc_weights.csv")

display(weight_df.style.format("{:.3f}").background_gradient(cmap="Blues").set_caption("Endpoint-wise Ensemble Weights — Validation Only"))

ensemble_summary, ensemble_per_endpoint = register_model_result(
    "Validation-weighted Soft-Voting Ensemble",
    P_train_ens, P_val_ens, P_test_ens,
    notes="Endpoint-wise soft voting; weights derived only from validation AUC with shrinkage. No test-label leakage."
)
save_runtime_session()


# Final Results and Visualizations

Visualization format `Ultimate_Final.ipynb`, `Final_Ultra.ipynb` এবং supplied screenshot style অনুসরণ করে clean, formal এবং research-friendly রাখা হয়েছে।


In [ ]:
if "SOHOJ_READY" not in globals():
    import dill
    from pathlib import Path
    session_file = Path(Path("sohoj_session_path.txt").read_text(encoding="utf-8").strip())
    print("♻️ Kernel restart detected; saved session load হচ্ছে...")
    import __main__
    dill.load_module(str(session_file), module=__main__)
    if "rehydrate_runtime" in globals():
        rehydrate_runtime()

def collect_results():
    summaries, per_endpoint = [], []
    for summary_path in OUTPUT_DIR.glob("*/overall_summary.json"):
        try:
            summary = json.loads(summary_path.read_text(encoding="utf-8"))
            summaries.append(summary)
            metric_path = summary_path.parent / "per_endpoint_metrics.csv"
            if metric_path.exists():
                temp = pd.read_csv(metric_path)
                temp["Model"] = summary["Model"]
                per_endpoint.append(temp)
        except Exception as exc:
            print("Skipped unreadable result:", summary_path, exc)
    summary_df = pd.DataFrame(summaries)
    if len(summary_df):
        summary_df = summary_df.sort_values("Test Mean AUC", ascending=False).reset_index(drop=True)
    endpoint_df = pd.concat(per_endpoint, ignore_index=True) if per_endpoint else pd.DataFrame()
    return summary_df, endpoint_df

summary_df, endpoint_results_df = collect_results()
summary_df.to_csv(OUTPUT_DIR / "final_model_leaderboard.csv", index=False)
endpoint_results_df.to_csv(OUTPUT_DIR / "final_endpoint_results.csv", index=False)

display(
    summary_df[["Model", "Train Mean AUC", "Validation Mean AUC", "Test Mean AUC", "Test Mean Accuracy", "Train-Validation AUC Gap", "Fitting Status", "Target AUC Reached"]]
    .style.format({
        "Train Mean AUC": "{:.4f}", "Validation Mean AUC": "{:.4f}",
        "Test Mean AUC": "{:.4f}", "Test Mean Accuracy": "{:.4f}",
        "Train-Validation AUC Gap": "{:+.4f}"
    })
    .background_gradient(subset=["Test Mean AUC"], cmap="RdYlGn", vmin=0.5, vmax=1.0)
    .set_caption("Final Model Leaderboard — Mean AUC and Mean Accuracy")
)

# Leaderboard tables-ও restart-safe session-এ রাখা হচ্ছে।
save_runtime_session()


In [ ]:
if "SOHOJ_READY" not in globals():
    import dill
    from pathlib import Path
    session_file = Path(Path("sohoj_session_path.txt").read_text(encoding="utf-8").strip())
    print("♻️ Kernel restart detected; saved session load হচ্ছে...")
    import __main__
    dill.load_module(str(session_file), module=__main__)
    if "rehydrate_runtime" in globals():
        rehydrate_runtime()

if len(summary_df):
    fig, axes = plt.subplots(1, 2, figsize=(18, 7))
    auc_df = summary_df.sort_values("Test Mean AUC")
    axes[0].barh(auc_df["Model"], auc_df["Test Mean AUC"], color="#2A9D8F")
    axes[0].axvline(TARGET_MEAN_AUC, color="#E76F51", linestyle="--", label=f"Target = {TARGET_MEAN_AUC:.2f}")
    axes[0].set_xlim(0.5, 1.02)
    axes[0].set_title("Model Comparison — Mean Test AUC", fontweight="bold")
    axes[0].set_xlabel("Mean AUC-ROC")
    axes[0].legend()
    for i, v in enumerate(auc_df["Test Mean AUC"]): axes[0].text(v + 0.006, i, f"{v:.3f}", va="center")

    acc_df = summary_df.sort_values("Test Mean Accuracy")
    axes[1].barh(acc_df["Model"], acc_df["Test Mean Accuracy"], color="#457B9D")
    axes[1].set_xlim(0.5, 1.02)
    axes[1].set_title("Model Comparison — Mean Test Accuracy", fontweight="bold")
    axes[1].set_xlabel("Mean Accuracy")
    for i, v in enumerate(acc_df["Test Mean Accuracy"]): axes[1].text(v + 0.006, i, f"{v:.3f}", va="center")

    plt.tight_layout()
    plt.savefig(FIGURE_DIR / "10_model_comparison_auc_accuracy.png", dpi=240, bbox_inches="tight")
    plt.show()


In [ ]:
if "SOHOJ_READY" not in globals():
    import dill
    from pathlib import Path
    session_file = Path(Path("sohoj_session_path.txt").read_text(encoding="utf-8").strip())
    print("♻️ Kernel restart detected; saved session load হচ্ছে...")
    import __main__
    dill.load_module(str(session_file), module=__main__)
    if "rehydrate_runtime" in globals():
        rehydrate_runtime()

if len(summary_df):
    plt.figure(figsize=(11, 7))
    statuses = summary_df["Fitting Status"].unique().tolist()
    palette = dict(zip(statuses, sns.color_palette("Set2", n_colors=max(3, len(statuses)))))
    for status, grp in summary_df.groupby("Fitting Status"):
        plt.scatter(grp["Train Mean AUC"], grp["Validation Mean AUC"], s=120, label=status, color=palette[status], edgecolor="black", alpha=0.9)
        for _, r in grp.iterrows():
            plt.annotate(r["Model"], (r["Train Mean AUC"], r["Validation Mean AUC"]), xytext=(5, 5), textcoords="offset points", fontsize=8)
    lo = min(0.5, summary_df[["Train Mean AUC", "Validation Mean AUC"]].min().min() - 0.02)
    plt.plot([lo, 1], [lo, 1], "--", color="gray", label="Train = Validation")
    plt.xlim(lo, 1.01); plt.ylim(lo, 1.01)
    plt.xlabel("Train Mean AUC"); plt.ylabel("Validation Mean AUC")
    plt.title("Overfitting / Underfitting Diagnostic", fontweight="bold")
    plt.legend(loc="best")
    plt.tight_layout()
    plt.savefig(FIGURE_DIR / "11_fitting_diagnostic.png", dpi=240, bbox_inches="tight")
    plt.show()


In [ ]:
if "SOHOJ_READY" not in globals():
    import dill
    from pathlib import Path
    session_file = Path(Path("sohoj_session_path.txt").read_text(encoding="utf-8").strip())
    print("♻️ Kernel restart detected; saved session load হচ্ছে...")
    import __main__
    dill.load_module(str(session_file), module=__main__)
    if "rehydrate_runtime" in globals():
        rehydrate_runtime()

if len(endpoint_results_df):
    auc_pivot = endpoint_results_df.pivot(index="Endpoint", columns="Model", values="Test AUC")
    plt.figure(figsize=(max(14, 1.25 * len(auc_pivot.columns)), 7))
    sns.heatmap(auc_pivot, annot=True, fmt=".2f", cmap="RdYlGn", vmin=0.5, vmax=1.0, linewidths=0.4, cbar_kws={"label": "Test AUC"})
    plt.title("AUC Heatmap — All Models × All Tox21 Endpoints", fontweight="bold")
    plt.xticks(rotation=35, ha="right")
    plt.tight_layout()
    plt.savefig(FIGURE_DIR / "12_auc_heatmap.png", dpi=240, bbox_inches="tight")
    plt.show()

    acc_pivot = endpoint_results_df.pivot(index="Endpoint", columns="Model", values="Test Accuracy")
    plt.figure(figsize=(max(14, 1.25 * len(acc_pivot.columns)), 7))
    sns.heatmap(acc_pivot, annot=True, fmt=".2f", cmap="YlGnBu", vmin=0.0, vmax=1.0, linewidths=0.4, cbar_kws={"label": "Test Accuracy"})
    plt.title("Accuracy Heatmap — All Models × All Tox21 Endpoints", fontweight="bold")
    plt.xticks(rotation=35, ha="right")
    plt.tight_layout()
    plt.savefig(FIGURE_DIR / "13_accuracy_heatmap.png", dpi=240, bbox_inches="tight")
    plt.show()


In [ ]:
if "SOHOJ_READY" not in globals():
    import dill
    from pathlib import Path
    session_file = Path(Path("sohoj_session_path.txt").read_text(encoding="utf-8").strip())
    print("♻️ Kernel restart detected; saved session load হচ্ছে...")
    import __main__
    dill.load_module(str(session_file), module=__main__)
    if "rehydrate_runtime" in globals():
        rehydrate_runtime()

if len(summary_df):
    best_model = summary_df.iloc[0]["Model"]
    best_rows = endpoint_results_df[endpoint_results_df["Model"] == best_model].copy()
    x = np.arange(len(best_rows))
    fig, axes = plt.subplots(1, 2, figsize=(18, 6))

    auc_colors = ["#2A9D8F" if v >= 0.85 else "#F4A261" if v >= 0.70 else "#E76F51" for v in best_rows["Test AUC"]]
    axes[0].bar(x, best_rows["Test AUC"], color=auc_colors)
    axes[0].axhline(best_rows["Test AUC"].mean(), linestyle="--", color="black", label=f"Mean={best_rows['Test AUC'].mean():.3f}")
    axes[0].set_ylim(0.5, 1.03); axes[0].set_title(f"AUC per Endpoint — {best_model}", fontweight="bold")
    axes[0].set_xticks(x); axes[0].set_xticklabels(best_rows["Endpoint"], rotation=45, ha="right"); axes[0].legend()

    axes[1].bar(x, best_rows["Test Accuracy"], color="#457B9D")
    axes[1].axhline(best_rows["Test Accuracy"].mean(), linestyle="--", color="black", label=f"Mean={best_rows['Test Accuracy'].mean():.3f}")
    axes[1].set_ylim(0.5, 1.03); axes[1].set_title(f"Accuracy per Endpoint — {best_model}", fontweight="bold")
    axes[1].set_xticks(x); axes[1].set_xticklabels(best_rows["Endpoint"], rotation=45, ha="right"); axes[1].legend()

    plt.tight_layout()
    plt.savefig(FIGURE_DIR / "14_best_model_per_endpoint.png", dpi=240, bbox_inches="tight")
    plt.show()
    print("✅ Best model by Mean Test AUC:", best_model)


In [ ]:
if "SOHOJ_READY" not in globals():
    import dill
    from pathlib import Path
    session_file = Path(Path("sohoj_session_path.txt").read_text(encoding="utf-8").strip())
    print("♻️ Kernel restart detected; saved session load হচ্ছে...")
    import __main__
    dill.load_module(str(session_file), module=__main__)
    if "rehydrate_runtime" in globals():
        rehydrate_runtime()

if len(summary_df):
    best_model = summary_df.iloc[0]["Model"]
    best_dir = OUTPUT_DIR / slugify(best_model)
    pred = np.load(best_dir / "predictions.npz")
    Pbest = pred["test"]
    best_rows = pd.read_csv(best_dir / "per_endpoint_metrics.csv")
    best_ep = best_rows.sort_values("Test AUC", ascending=False).iloc[0]["Endpoint"]
    hardest_ep = endpoint_summary_raw.sort_values("Positive Rate %").iloc[0]["Endpoint"]

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    for ax, target in zip(axes, [best_ep, hardest_ep]):
        j = TARGETS.index(target)
        mask = np.isfinite(Y_test[:, j])
        cm = confusion_matrix(Y_test[mask, j].astype(int), (Pbest[mask, j] >= THRESHOLD).astype(int), labels=[0, 1])
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax, xticklabels=["Non-Toxic", "Toxic"], yticklabels=["Non-Toxic", "Toxic"])
        ax.set_title(f"{target}\nAUC={safe_auc(Y_test[:, j], Pbest[:, j]):.3f}", fontweight="bold")
        ax.set_xlabel("Predicted label"); ax.set_ylabel("True label")
    fig.suptitle(f"Confusion Matrices — Best vs Hardest Endpoint ({best_model})", fontweight="bold", fontsize=14)
    plt.tight_layout()
    plt.savefig(FIGURE_DIR / "15_confusion_matrices.png", dpi=240, bbox_inches="tight")
    plt.show()


In [ ]:
if "SOHOJ_READY" not in globals():
    import dill
    from pathlib import Path
    session_file = Path(Path("sohoj_session_path.txt").read_text(encoding="utf-8").strip())
    print("♻️ Kernel restart detected; saved session load হচ্ছে...")
    import __main__
    dill.load_module(str(session_file), module=__main__)
    if "rehydrate_runtime" in globals():
        rehydrate_runtime()

# Training curves for all completed neural models.
curve_files = []
for history_path in OUTPUT_DIR.glob("*/history.json"):
    try:
        h = json.loads(history_path.read_text(encoding="utf-8"))
        if h.get("train_loss") and h.get("val_loss"):
            curve_files.append((history_path.parent.name, h))
    except Exception:
        pass

if curve_files:
    n = len(curve_files)
    fig, axes = plt.subplots(n, 1, figsize=(12, max(4, 3.5*n)), squeeze=False)
    for ax, (slug, h) in zip(axes.flat, curve_files):
        ax.plot(h["train_loss"], label="Train Loss", linewidth=2)
        ax.plot(h["val_loss"], label="Validation Loss", linewidth=2)
        ax.set_title(slug.replace("_", " ").title(), fontweight="bold")
        ax.set_xlabel("Epoch"); ax.set_ylabel("Loss"); ax.legend(); ax.grid(alpha=0.25)
    plt.tight_layout()
    plt.savefig(FIGURE_DIR / "16_neural_training_curves.png", dpi=240, bbox_inches="tight")
    plt.show()
else:
    print("ℹ️ Neural model history এখনো পাওয়া যায়নি।")


In [ ]:
if "SOHOJ_READY" not in globals():
    import dill
    from pathlib import Path
    session_file = Path(Path("sohoj_session_path.txt").read_text(encoding="utf-8").strip())
    print("♻️ Kernel restart detected; saved session load হচ্ছে...")
    import __main__
    dill.load_module(str(session_file), module=__main__)
    if "rehydrate_runtime" in globals():
        rehydrate_runtime()

# Classical 3-fold CV summary.
cv_frames = []
for cv_path in OUTPUT_DIR.glob("*/three_fold_cv.csv"):
    try: cv_frames.append(pd.read_csv(cv_path))
    except Exception: pass

if cv_frames:
    cv_all = pd.concat(cv_frames, ignore_index=True)
    cv_summary = cv_all.groupby("Model").agg(
        CV_Mean_AUC=("CV Mean AUC", "mean"),
        CV_Std_AUC=("CV Mean AUC", "std"),
        CV_Mean_Accuracy=("CV Mean Accuracy", "mean")
    ).reset_index().sort_values("CV_Mean_AUC", ascending=False)
    display(cv_summary.style.format({"CV_Mean_AUC": "{:.4f}", "CV_Std_AUC": "{:.4f}", "CV_Mean_Accuracy": "{:.4f}"}).set_caption("Classical Models — 3-Fold Cross-Validation"))

    plt.figure(figsize=(11, 5))
    x = np.arange(len(cv_summary))
    plt.bar(x, cv_summary["CV_Mean_AUC"], yerr=cv_summary["CV_Std_AUC"].fillna(0), capsize=5, color="#457B9D")
    plt.xticks(x, cv_summary["Model"], rotation=25, ha="right")
    plt.ylim(0.5, 1.02); plt.ylabel("Mean CV AUC"); plt.title("3-Fold Cross-Validation — Mean AUC", fontweight="bold")
    for i, v in enumerate(cv_summary["CV_Mean_AUC"]): plt.text(i, v + 0.015, f"{v:.3f}", ha="center")
    plt.tight_layout()
    plt.savefig(FIGURE_DIR / "17_classical_three_fold_cv.png", dpi=240, bbox_inches="tight")
    plt.show()


## Final export, reproducibility report and research notes


In [ ]:
if "SOHOJ_READY" not in globals():
    import dill
    from pathlib import Path
    session_file = Path(Path("sohoj_session_path.txt").read_text(encoding="utf-8").strip())
    print("♻️ Kernel restart detected; saved session load হচ্ছে...")
    import __main__
    dill.load_module(str(session_file), module=__main__)
    if "rehydrate_runtime" in globals():
        rehydrate_runtime()

import sklearn, rdkit, xgboost

package_versions = {
    "python": sys.version,
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "scikit-learn": sklearn.__version__,
    "rdkit": rdkit.__version__,
    "xgboost": xgboost.__version__,
    "torch": torch.__version__,
    "torchvision_available": TORCHVISION_AVAILABLE,
    "device": str(DEVICE),
    "gpu": torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else None,
    "ram_gb": round(RAM_GB, 2),
    "resource_mode": RESOURCE_MODE,
}

run_config = {
    "notebook": "Sohoj.ipynb",
    "dataset_path": str(DATA_PATH),
    "dataset_sha256": RAW_SHA256,
    "cleaned_samples": len(df),
    "split": {"train": 0.70, "validation": 0.10, "test": 0.20},
    "seed": SEED,
    "metrics": ["Mean AUC-ROC", "Mean Accuracy"],
    "threshold": THRESHOLD,
    "target_mean_auc": TARGET_MEAN_AUC,
    "missing_label_strategy": "mask/ignore; never impute target labels as 0 or 1",
    "rgcn_removed": True,
    "packages": package_versions,
    "completed_models": summary_df["Model"].tolist() if len(summary_df) else [],
    "generated_at": datetime.now().isoformat(timespec="seconds"),
}

(OUTPUT_DIR / "run_config_and_environment.json").write_text(json.dumps(run_config, indent=2), encoding="utf-8")
endpoint_summary_raw.to_csv(OUTPUT_DIR / "endpoint_missing_imbalance_summary.csv", index=False)
split_dist.to_csv(OUTPUT_DIR / "split_endpoint_distribution.csv", index=False)

print("=" * 88)
print("✅ Sohoj pipeline export complete।")
print("Final leaderboard:", OUTPUT_DIR / "final_model_leaderboard.csv")
print("Endpoint results  :", OUTPUT_DIR / "final_endpoint_results.csv")
print("Figures folder    :", FIGURE_DIR)
print("Models folder     :", MODEL_DIR)
print("Checkpoints folder:", CHECKPOINT_DIR)
print("=" * 88)

if len(summary_df):
    best = summary_df.iloc[0]
    print(f"Best model: {best['Model']}")
    print(f"Mean Test AUC: {best['Test Mean AUC']:.4f}")
    print(f"Mean Test Accuracy: {best['Test Mean Accuracy']:.4f}")
    if best["Test Mean AUC"] >= TARGET_MEAN_AUC:
        print("🎯 Target Mean AUC ≥ 0.90 achieved।")
    else:
        print("ℹ️ Target Mean AUC 0.90 এখনো achieved হয়নি; result fabricate করা হয়নি।")


## Research implementation notes

1. **RF, Extra Trees, XGBoost**: `Ultimate_Final.ipynb`-এর standardization, canonicalization, duplicate removal, Morgan ECFP4, MACCS, all RDKit descriptors, train-only median imputation/scaling, model parameters এবং 3-fold CV preserved। শুধু user-requested 70/10/20 split যোগ করা হয়েছে।
2. **SVM Tanimoto**: DeepTox-এর ECFP4 similarity-kernel concept-এর open-source exact binary Tanimoto implementation।
3. **DeepTox MT-DNN**: paper-এর masked multi-task logistic learning, ReLU, dropout, SGD, L2 and early stopping principles রাখা হয়েছে; proprietary descriptor stack-এর বদলে reproducible RDKit stack।
4. **CapsNet**: 179 inputs, 256/128 RBMs, 8×8 PrimaryCaps, two 2D class capsules, 2 routing iterations এবং margin loss implemented। Missing experimental LogD/solubility-এর জন্য documented proxies।
5. **DenseNet121 + SVM Tanimoto**: user-requested hybrid; supplied papers-এ exact architecture ছিল না।
6. **Multi-channel CNN**: 24 Å/0.5 Å six-channel grid, four-stage CNN, BN/dropout এবং paper-inspired imbalance handling। AutoDock/ChemAxon parameters unavailable হওয়ায় transparent RDKit approximation।
7. **MTL-DNN**: 2055 inputs, 1024→512→128, dropout 0.3, Adam 1e-4, class-weighted masked BCE।
8. **Ensemble**: validation-only endpoint weights; test leakage নেই।
9. **No R-GCN**: notebook-এ R-GCN code, dependency বা result নেই।
10. **No fake output**: AUC/Accuracy run-time-এ actual predictions থেকে compute হয়।


## References from the supplied papers

1. Mayr, A., Klambauer, G., Unterthiner, T., & Hochreiter, S. (2016). *DeepTox: Toxicity Prediction using Deep Learning*. Frontiers in Environmental Science, 3:80.
2. Wang, Y. et al. (2021). *Multitask CapsNet: An Imbalanced Data Deep Learning Method for Predicting Toxicants*. ACS Omega, 6, 26545–26555.
3. Yuan, Q. et al. (2019). *Toxicity Prediction Method Based on Multi-Channel Convolutional Neural Network*. Molecules, 24, 3383.
4. Narayana, V. L. et al. (2026). *Deep Learning-Based Toxicity Prediction of Drug Compounds Using Multi-Task Neural Networks on the Tox21 Dataset*.
5. The additional uploaded GCN/knowledge-graph papers were reviewed for preprocessing and evaluation context; **R-GCN was intentionally excluded** per requirement.
